In [ ]:
import subprocess
import sys

# Esse kernel so PRODUZ um checkpoint pra inspecao local -- nao e o que
# vai ser submetido direto (isso fica pro notebook de inferencia final,
# que roda em CPU ou GPU T4 selecionada manualmente). Entao P100 aqui nao
# e um problema de regra, mas o torch pre-instalado no Kaggle so suporta
# sm_70+ -- se vier P100 (sm_60), o forward quebra. Mesma correcao de
# sempre: build cu118, que cobre GPUs mais antigas tambem.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "torch==2.5.1", "torchvision==0.20.1",
        "--index-url", "https://download.pytorch.org/whl/cu118",
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "transformers==4.46.3"],
    check=True,
)


In [ ]:
from pathlib import Path
from IPython.display import Image, display


def _find_cover(name="RSNA_KNEE_1.png"):
    """Localiza a imagem de capa onde quer que o dataset dela tenha sido montado.

    Um caminho de montagem fixo protegido por `exists()` é o pior dos dois mundos:
    se errar, a imagem simplesmente não está lá, sem nada ser dito. O Kaggle
    também nem sempre monta um dataset na mesma profundidade. Buscar nos inputs
    anexados custa uma listagem de diretório por input e não pode falhar
    silenciosamente. A montagem da competição é pulada por inspeção em vez de
    por nome, porque ela contém centenas de milhares de arquivos e nenhum deles
    é este.
    """
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    for d in sorted(p for p in base.iterdir() if p.is_dir()):
        if any((d / s).is_dir() for s in ("train_series", "test_series")):
            continue
        hit = next(d.rglob(name), None)
        if hit is not None:
            return hit
    return None


_cover = _find_cover()
if _cover is not None:
    display(Image(filename=str(_cover)))


# Doze achados a partir de uma ressonância de joelho

Cada estudo é um conjunto de séries de ressonância adquiridas numa sessão, e a tarefa é
atribuir a ele doze probabilidades: lesão do ligamento cruzado anterior e do colateral
medial, ruptura meniscal medial e lateral, osteoartrite em cada um dos três
compartimentos, efusão articular, sinovite, cisto de Baker, contusão óssea e fratura.

As seções estão ordenadas pela forma como as decisões se restringem entre si: o que a
métrica recompensa decide como as previsões são combinadas, de onde vêm os alvos decide
o que pode ser treinado, e o que o aparelho registrou decide o que é mostrado ao
encoder.


> **Sobre as duas fontes de labels.** O §2 descreve dois leitores para uma mesma tarefa:
> um extrator baseado em regras, definido por completo abaixo, e um modelo de linguagem
> que lê os mesmos laudos, cuja saída é uma tabela pública anexada como dataset. Com a
> tabela montada, ela fornece os alvos; sem ela, o extrator fornece; e todas as células
> depois dessa permanecem inalteradas.


## 1. O que a métrica recompensa

A métrica é a média não ponderada de doze AUC-ROC por label:

$$\text{Score} \;=\; \frac{1}{12}\sum_{i=0}^{11} \mathrm{AUC}_i .$$

Três consequências decorrem disso, e cada uma elimina uma decisão de design.

**Só a ordem importa.** $\mathrm{AUC}_i$ é invariante sob qualquer mapeamento
estritamente crescente dos scores do label $i$, então calibração e limiares não valem
nada. Isso também fixa como combinar modelos: fazer a média das probabilidades brutas
deixa o modelo mais confiante dominar, enquanto fazer a média dos *ranks* combina a
única informação que a métrica lê.

**Todo label custa o mesmo.** Seja $M$ a AUC média que um bom modelo conseguiria
atingir. Um label deixado no acaso contribui com $0.5$ em vez de aproximadamente $M$,
abrindo mão de $(M-0.5)/12$ do score final por melhor que os outros onze se saiam — com
$M = 0.85$, isso são $0.029$. Achados raros merecem *mais* atenção do que os comuns,
porque um achado raro é onde um modelo mais facilmente acaba no acaso.

**Deslocamentos de prevalência são sobreviveis, limiares não são.** A AUC é, em
expectativa, invariante à taxa de positivos, e a competição afirma que a prevalência não
tem garantia de coincidir entre os conjuntos de treino, público e final. Um único corte
sobrevive abaixo — os alvos graduados são binarizados no ponto médio para que uma AUC em
holdout possa sequer ser calculada — e ele decide qual época e configuração são
mantidas, nunca um score submetido.


## 2. De onde vêm os alvos

Apenas um pequeno subconjunto dos estudos de treino carrega os doze labels por
condição. Todo estudo de treino carrega o laudo radiológico original, e a descrição dos
dados convida a derivar labels a partir dele.

O fato decisivo está nos esquemas, não na prosa: `train.csv` tem uma coluna `Report` e
`test.csv` não tem. O texto está disponível no ajuste e ausente na predição. Isso
descarta um modelo de fusão com um branch de texto — na inferência ele não teria nada
pra ler — e deixa os laudos utilizáveis só como alvos de treino, como sinal auxiliar
descartado na inferência, ou como um peso sobre o quão confiantemente um estudo pôde ser
lido. Este notebook adota a primeira e a terceira opções: um extrator de regras
multilíngue lê cada laudo cláusula por cláusula, decidindo para cada achado se a
cláusula afirma, nega ou hedgeia (deixa em dúvida) ele, e emite um score com uma
confiança. A confiança vira um peso de amostra, então um estudo cujo laudo não diz nada
sobre sinovite puxa muito menos naquela cabeça do que um que a nomeia.

**Dois leitores.** Um léxico casa morfologia, então seu modo de falha é silêncio em vez
de erro, e silêncio é mensurável sem gabarito: para cada par (laudo, achado), pergunta
só se algo casou. Essa taxa não precisa de anotações, então roda em todo estudo, e
marcada por idioma ela diz *onde* o vocabulário é raso. Aqui as falhas se concentram —
um idioma é coberto muito melhor que os outros oito, e a lacuna cai sobre achados que um
laudo de joelho quase sempre comenta. Enumerar morfologia pra nove idiomas é o
instrumento errado pra isso; ler a frase é o certo, e um modelo de linguagem a lê,
questionado sobre os mesmos doze achados na mesma forma graduada. Os estudos anotados
decidem entre os leitores, pareados nos mesmos estudos, e a diferença é grande e
unilateral na direção que a taxa de cobertura prevê. Então o pipeline prefere uma tabela
montada de labels lidos por modelo quando presente e roda o léxico quando não; ambos
emitem as mesmas colunas, e uma tabela parcial recorre ao fallback por estudo, não por
execução.

Dois detalhes importam mais que os internos de qualquer um dos leitores.

**Laudos são graduados, anotações são limiarizadas.** O radiologista que escreve o laudo
e o anotador não compartilham um limiar. Um laudo dizendo *pequena efusão articular*
pode estar contra uma anotação negativa, porque o anotador marcou só efusões que julgou
significativas. Uma regra do tipo *termo presente $\Rightarrow$ positivo* está errada por
construção; graduar a menção — traço, sem qualificador, marcada — está certo e não custa
nada, porque o §1 estabeleceu que só a ordem é lida.

**Labels derivados não são independentes entre estudos.** Um laudo compartilhado
literalmente por vários estudos gera um único vetor de alvo pra todos eles, o que tem
que ser respeitado ao dividir; o §7 faz isso.


### Lendo um laudo em nove idiomas

**Nenhum idioma é identificado.** Todo léxico de pistas carrega os nove idiomas de uma
vez e cada cláusula é testada contra a união. Rotear primeiro significa se comprometer
com um palpite antes de qualquer evidência ser lida, e o palpite barato — testes de
substring, `'the '` para inglês, `'la '` para francês — falha feio, porque `la` é tão
comum em espanhol quanto em francês e qualquer teste que rode primeiro engole os dois.
Agrupar custa pouco, já que pistas em grego e cirílico não podem colidir com as de
script latino e os vocabulários de script latino de interesse são próximos o bastante
pra que uma pista compartilhada normalmente esteja certa. O preço é pago em cobertura em
vez disso.

**Normaliza, depois segmenta, depois delimita o escopo.** Caixa, diacríticos e
separadores são achatados primeiro, o que também conserta um problema de codepoint:
muitos laudos gregos escrevem mu com MICRO SIGN U+00B5 em vez de U+03BC, e NFKD mapeia
um no outro. O texto é então dividido em cláusulas, com uma linha de cabeçalho anexada
ao valor abaixo dela, porque um laudo que lê `Fractures :` e depois `Aucune.` afirma uma
coisa só em duas linhas, e qualquer método que as separe lê uma negação como um
positivo.

**Afirmação, negação, dúvida.** Negação não é caso de borda: para vários achados a
maioria das menções é negativa, já que um laudo lista o que foi checado e encontrado
íntegro. Normalidade explícita conta como negação — *ligamentos cruzados y colaterales
dentro de límites normales* é evidência de ausência, não ausência de evidência — exceto
onde uma ruptura ou um grau alto é citado na mesma frase.

**Radicais por trás de frases.** Quatro alvos precisam de uma palavra de anatomia e uma
palavra de patologia juntas. Um léxico de frases completas carrega a maioria dos casos,
mas não sobrevive à morfologia: o turco sufixa possessivos no substantivo, o croata e o
grego o declinam. Onde a frase falha, uma segunda passada casa um radical e exige um
qualificador de lado dentro de uma janela de *caracteres*, o que lida com flexão sem
enumerá-la, e lida com ordem de palavras — que coloca o adjetivo de lado antes do
substantivo em inglês e depois dele em grego.

**Por que cobertura decide e concordância não.** Uma regra que nunca dispara não gera
erro nenhum; num extrator binário ela emite um negativo, indistinguível de um confiante,
e um léxico completo em inglês e raso em grego parece um corpus onde pacientes gregos
têm menos achados. A checagem óbvia — concordância com as anotações por condição — mede
a coisa certa em estudos de menos demais. O erro-padrão de Hanley–McNeil de uma AUC $A$
com $n_p$ positivos e $n_n$ negativos,

$$
\mathrm{SE}(A)=\sqrt{\frac{A(1-A)+(n_p-1)(Q_1-A^{2})+(n_n-1)(Q_2-A^{2})}{n_p\,n_n}},
\qquad
Q_1=\frac{A}{2-A},\quad Q_2=\frac{2A^{2}}{1+A},
$$

em $A\approx0.8$ e um punhado de positivos entre algumas dezenas de estudos chega perto
de $0.09$ — um intervalo de 95% de mais ou menos $\pm0.17$ — bem mais largo do que as
diferenças entre dois léxicos que ela seria chamada a separar. A taxa de silêncio não
tem esse limite, então é ela que aponta para o vocabulário faltante e é nela que
mudanças no léxico são julgadas.


In [ ]:
from __future__ import annotations

import re
import unicodedata

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]


# O i com/sem ponto do turco precisa ser achatado antes do casefolding, senão
# "İZLENMEZ" e "izlenmez" divergem. O ß e o d-com-traço croata/sérvio idem.
_PRE = str.maketrans({
    "ı": "i", "İ": "i", "I": "i", "ß": "ss", "đ": "d", "Đ": "d",
    "ø": "o", "Ø": "o", "æ": "ae", "Æ": "ae",
})


def normalize(text: str) -> str:
    """Achata caixa, diacríticos e separadores; mantém letras gregas e cirílicas.

    A decomposição NFKD remove acentos latinos e o tonos grego igualmente (ά -> α), que é
    o que queremos: laudos são inconsistentes quanto a acentos. Também mapeia o MICRO SIGN
    U+00B5 pra um mu de verdade, o que importa porque a maioria dos laudos gregos aqui usa
    o codepoint errado.
    """
    if not isinstance(text, str):
        return ""
    text = text.translate(_PRE).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace("­", "")                    # hífen suave (soft hyphen)
    text = re.sub(r"[_\-/\\]+", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text


_SENT_SPLIT = re.compile(r"(?<=[.;!?])\s+|\n+")


def clauses(text: str):
    """Divide em cláusulas, depois anexa linhas de `cabeçalho:` ao valor que vem a seguir.

    Uma linha de laudo lendo `Fractures :` seguida de `Aucune.` é uma única afirmação.
    Dividir só por pontuação separa a anatomia da sua negação e inverte o label.
    """
    norm = normalize(text)
    raw = [c.strip() for c in _SENT_SPLIT.split(norm) if c and c.strip()]

    merged = []
    for i, c in enumerate(raw):
        # Um fragmento terminando em dois-pontos é um cabeçalho para o próximo fragmento.
        # Laudos estruturados em inglês escrevem cabeçalhos longos - "lateral compartment
        # (meniscus, collateral ligament complex, cartilage):" tem oito palavras - então o
        # teto é generoso.
        #
        # Um cabeçalho mesclado NÃO pode também ficar sozinho. Sozinho, ele carrega a
        # palavra de anatomia sem nenhuma negação no escopo, então `Fractures :` /
        # `Aucune.` afirmaria uma fratura pelo cabeçalho isolado enquanto a cláusula unida
        # lê corretamente a negação. A cláusula unida é um superconjunto do cabeçalho,
        # então nada se perde ao descartá-lo; um cabeçalho sem valor abaixo dele não é
        # mesclado e continua valendo sozinho.
        if c.endswith(":") and len(c.split()) <= 14 and i + 1 < len(raw):
            merged.append(c + " " + raw[i + 1])
        else:
            merged.append(c)
    # Enumerações separadas por vírgula dentro de uma cláusula longa escondem
    # afirmações separadas.
    out = []
    for c in merged:
        out.append(c)
        if len(c.split()) > 25:
            out.extend(p.strip() for p in c.split(",") if len(p.split()) > 2)
    return out


def _rx(*alts: str) -> re.Pattern:
    return re.compile("|".join(alts))


In [ ]:
NEGATION = _rx(
    # en
    r"\bno\b", r"\bnot\b", r"\bwithout\b", r"\bnegative for\b", r"\babsence\b",
    r"\bno evidence\b", r"\bunremarkable\b", r"\bfree of\b", r"\bnone\b", r"\bnil\b",
    # es
    r"\bsin\b", r"\bno hay\b", r"\bausencia\b", r"\bausentes?\b",
    # fr
    r"\bpas de\b", r"\bsans\b", r"\baucune?\b", r"\babsence\b",
    # nl
    r"\bgeen\b", r"\bzonder\b", r"\bniet\b",
    # de
    r"\bkeine?\b", r"\bohne\b", r"\bnicht\b",
    # tr
    r"\byok\b", r"\byoktur\b", r"izlenmemekte", r"saptanmadi", r"\bdegil\b",
    r"gozlenmemekte", r"mevcut degil", r"eslik etmiyor", r"\bizlenmedi\b",
    # hr / sr / bs
    r"\bnema\b", r"\bbez\b", r"\bnisu\b", r"\bnije\b",
    # el (accents already stripped)
    r"\bδεν\b", r"\bχωρις\b", r"ουδεν",
    # bg / ru
    r"\bбез\b", r"\bне\b", r"липсва", r"\bняма\b",
)

NORMALITY = _rx(
    r"\bnormal", r"\bintact\b", r"\bpreserved\b", r"\bwithin normal limits\b",
    r"limites normales", r"\bconservad", r"\bintegr", r"\bnormales\b",
    r"\bdoga(l|ll)\b", r"korunmus", r"\bnormaldir\b", r"olagan",
    r"\buredn", r"\bocuvan", r"\bodrzan", r"\bintakt",
    r"φυσιολογικ", r"ακεραι",
    r"unauffallig", r"regelrecht", r"\bintakt\b",
    r"нормал", r"запазен", r"съхранен", r"\bбез особености\b",
    r"\bgaaf\b", r"\bnormaal\b",
)

UNCERTAIN = _rx(
    r"\bpossible\b", r"\bprobable\b", r"\bsuspicious\b", r"\bsuspected\b",
    r"cannot (be )?exclude", r"\bmay\b", r"\bquestionable\b", r"\bequivocal\b",
    r"\bposible\b", r"sin criterios categoricos", r"\bdudos",
    r"\bmuhtemel\b", r"\bolasi\b", r"\bsupheli\b", r"\bizlenim",
    r"\bmoguce\b", r"\bvjerojatno\b", r"\bsumnja\b",
    r"πιθαν", r"υποπτ",
    r"\bmoglich", r"\bverdachtig", r"\bfraglich", r"\bV\.a\.\b",
    r"\bвъзможно\b", r"\bвероятно\b", r"суспект",
    r"\bmogelijk\b", r"\bverdacht\b",
)

# Vocabulário de patologia compartilhado pelas regras pareadas.
TEAR = _rx(
    r"\btear", r"\btorn\b", r"\brupture", r"\bdisruption\b", r"discontinuit",
    r"\bavuls",
    r"\brotura\b", r"\broturas\b", r"\bruptura", r"\bdesgarro", r"\broto\b",
    r"\bdechirure", r"\bdechire",
    r"\bscheur", r"\bruptuur", r"gescheurd",
    r"riss(bildung|e|es)?\b", r"einriss", r"\bruptur", r"zerreiss", r"\blasion",
    r"\byirtik", r"\byirtig", r"\bkopma\b", r"butunluk kaybi", r"\brupturu\b",
    r"\bpuknuce", r"\bruptur", r"\bprekid\b", r"\bpukotin",
    r"ρηξη", r"ρηξις", r"ρηγμα",
    r"руптура", r"разкъсв", r"разрив", r"скъсв",
)

DEGEN = _rx(
    r"degenerat", r"\bmucoid\b", r"\bmyxoid\b", r"\bfray", r"\bfissur",
    r"dejeneratif", r"\bmukoid\b", r"degenerativn", r"εκφυλ", r"дегенерат",
    r"\bμυξοειδ", r"\bμυξωδ",
    r"\bmuco ?ide\b", r"aufgefasert",
)

INJURY = _rx(
    r"\binjur", r"\bsprain", r"\blesion", r"\blasion", r"\bedema\b", r"\boedema\b",
    r"\bodem\b", r"\bedem\b", r"\bοιδημα", r"\bодем", r"\bедем", r"\bstrain\b",
    r"\bhigh signal\b", r"\bsignal alteration\b", r"\bhiperintens", r"\bhyperintens",
    r"aumento de senal", r"alteracion de senal", r"cambio de senal",
    r"\bsignalanhebung", r"\bsignalalteration", r"verhoogd signaal", r"sinyal artis",
    r"αυξημενο σημα", r"повишен сигнал",
    r"\bthicken", r"\bzadebljanje\b", r"\bverdikking\b", r"\bdistenzij",
    r"\blaksite\b", r"\blaxity\b", r"\bpartial\b", r"\bparcijaln", r"\bparcial",
    r"\bpartiel", r"\bpartiell",
)


In [ ]:
ANAT = {
    "ACL": _rx(
        r"anterior cruciate", r"\bacl\b",
        r"cruzado anterior", r"\blca\b",
        r"croise anterieur",
        r"voorste kruisband", r"\bvkb\b",
        r"vorderes kreuzband", r"vorderen kreuzband", r"vordere kreuzband",
        r"on capraz", r"\bocb\b",
        r"prednji krizni", r"prednjeg krizn",
        r"προσθι[οα][^ ]* χιαστ", r"προσθιου χιαστου", r"χιαστο[^ ]* συνδεσμ",
        # "χιαστοι και πλαγιοι συνδεσμοι" separa o adjetivo do substantivo, então o
        # radical do adjetivo tem que ficar sozinho. O grego marca cruzado com ele sem
        # ambiguidade.
        r"\bχιαστ\w*",
        r"предна кръстна", r"предната кръстна",
        # Plural, sem qualificador: laudos rotineiramente liberam os dois cruzados numa
        # única cláusula ("Ligamentos cruzados y colaterales dentro de limites
        # normales"), então a forma no plural tem que casar sem qualificador de lado, ou
        # a cláusula inteira se perde.
        r"cruciate ligaments", r"ligamentos cruzados", r"ligaments croises",
        r"kruisbanden", r"kreuzbander", r"capraz baglar", r"krizn[a-z]* ligament[a-z]*",
        r"χιαστοι συνδεσμ", r"χιαστων συνδεσμ", r"кръстните връзки", r"кръстни връзки",
    ),
    "MCL": _rx(
        r"medial collateral", r"\bmcl\b", r"tibial collateral",
        r"colateral medial", r"colateral interno", r"\blcm\b",
        r"collateral medial", r"collateral interne",
        r"mediale collaterale", r"binnenband", r"\b(mediale|laterale) banden\b",
        r"\bcollaterale banden\b",
        r"innenband", r"mediales? kollateral",
        r"\bic yan bag", r"medial kollateral", r"\biyb\b",
        r"medijalni kolateraln", r"medijalnog kolateraln",
        r"εσω πλαγι", r"εσωτερικο πλαγι", r"\bπλαγι\w* συνδεσμ", r"\bπλαγιοι\b",
        r"медиален колатерал", r"вътрешна странична", r"\bколатерал\w*",
        # Mesmo padrão de plural dos cruzados.
        # "Ligamentos cruzados y colaterales" separa o substantivo do adjetivo, então o
        # adjetivo tem que ficar sozinho como pista.
        r"\bcolaterales\b", r"\bcollateraux\b", r"\bcollateralen\b", r"\bkolateralni\b",
        r"collateral ligaments", r"ligamentos colaterales", r"ligaments collateraux",
        r"collaterale banden", r"kollateralbander", r"seitenbander", r"yan baglar",
        r"kolateraln[a-z]* ligament[a-z]*", r"πλαγιοι συνδεσμ", r"πλαγιων συνδεσμ",
        r"колатерални връзки", r"страничните връзки",
    ),
    "Medial Meniscus": _rx(
        r"medial meniscus", r"\bmm\b(?= tear)", r"medial menisc",
        r"menisco medial", r"menisco interno",
        r"menisque medial", r"menisque interne",
        r"mediale meniscus", r"binnenmeniscus",
        r"innenmeniskus", r"medialen? meniskus", r"innenmeniskushinterhorn",
        r"medyal menisk", r"\bic menisk",
        r"medijalni meniskus", r"medijalnog meniskusa", r"medijalnom meniskusu",
        r"εσω μηνισκ", r"μηνισκ[^ ]* του εσω", r"εσω διαμερισμα[^.]{0,40}μηνισκ",
        r"медиалния менискус", r"медиален менискус", r"вътрешния менискус",
    ),
    "Lateral Meniscus": _rx(
        r"lateral meniscus", r"lateral menisc",
        r"menisco lateral", r"menisco externo",
        r"menisque lateral", r"menisque externe",
        r"laterale meniscus", r"buitenmeniscus",
        r"aussenmeniskus", r"lateralen? meniskus",
        r"lateral menisk", r"\bdis menisk",
        r"lateralni meniskus", r"lateralnog meniskusa", r"lateralnom meniskusu",
        r"εξω μηνισκ", r"μηνισκ[^ ]* του εξω", r"εξω διαμερισμα[^.]{0,40}μηνισκ",
        r"латералния менискус", r"латерален менискус", r"външния менискус",
    ),
}

# Osteoartrite raramente é escrita como "osteoarthritis". É escrita como perda de
# cartilagem, grau de condropatia, estreitamento do espaço articular, ou osteófitos -
# delimitada a um compartimento.
OA_EVIDENCE = _rx(
    r"osteoarthrit", r"\barthros", r"\bgonarthros", r"\bosteoarthros",
    r"chondropath", r"chondromalac", r"condropat", r"condromalac",
    r"cartilage loss", r"cartilage thinning", r"chondral (loss|defect|ulcer|thinning)",
    r"osteophyt", r"osteofit", r"osteofyt", r"osteofito", r"osteophyten",
    r"joint space narrowing", r"pinzamiento articular",
    r"kikirdak kayb", r"kikirdak incelme", r"kondropati", r"kondral",
    r"kraakbeen(lijden|verlies)", r"gonartrose", r"artrose",
    r"knorpel(verlust|schaden|defekt)", r"arthrose", r"gonarthrose",
    r"hrskavic", r"hondromalac", r"artroz", r"osteoartrit",
    r"χονδρ[^ ]*παθ", r"αρθριτ", r"αρθρωσ", r"οστεοφυτ",
    r"αρθρικου χονδρου", r"εξαλειψη του αρθρικου χονδρου",
    r"артроз", r"хондропат", r"остеофит", r"хрущял[^.]{0,30}(изтън|увред|дефект)",
    r"ulcera[s]? condral", r"cartilago[^.]{0,25}(perdida|adelgaz)",
    r"icrs grade", r"outerbridge",
)

COMPARTMENT = {
    "Medial OA": _rx(
        r"medial (femorotibial|tibiofemoral|compartment)",
        r"compartimento femorotibial medial", r"femorotibial interno",
        r"mediaal femorotibiaal", r"mediale femorotibial",
        r"medial femorotibial", r"medialen kompartiment", r"innere[sn]? kompartiment",
        r"medyal femorotibial", r"ic kompartman", r"medyal kompartman",
        r"medijaln[^ ]* (femorotibi|odjelj|kompartm)",
        r"εσω διαμερισμα", r"εσω κνημιαι", r"εσω μηριαι",
        r"медиалн[^ ]* (компартм|отдел|тибиал|феморотиб)",
        r"medial (femoral|tibial) (condyle|plateau)", r"condilo femoral medial",
        r"medialen? (femurkondyl|tibiaplateau)", r"mediale femorale condyl",
    ),
    "Lateral OA": _rx(
        r"lateral (femorotibial|tibiofemoral|compartment)",
        r"compartimento femorotibial lateral", r"femorotibial externo",
        r"lateraal femorotibiaal", r"laterale femorotibial",
        r"lateral femorotibial", r"lateralen kompartiment", r"aussere[sn]? kompartiment",
        r"lateral femorotibial", r"dis kompartman", r"lateral kompartman",
        r"lateraln[^ ]* (femorotibi|odjelj|kompartm)",
        r"εξω διαμερισμα", r"εξω κνημιαι", r"εξω μηριαι",
        r"латералн[^ ]* (компартм|отдел|тибиал|феморотиб)",
        r"lateral (femoral|tibial) (condyle|plateau)", r"condilo femoral lateral",
        r"lateralen? (femurkondyl|tibiaplateau)", r"laterale femorale condyl",
    ),
    "PF OA": _rx(
        r"patellofemoral", r"femoropatellar", r"femoropatelar", r"patelofemoral",
        r"retropatellar", r"retrorotulian", r"\btrochlea", r"\btroclea", r"\btroklea",
        r"\bpatella\b", r"\bpatellar\b", r"\brotulian", r"\brotula\b", r"\bpatele\b",
        r"\bpatellae?\b", r"patellofemoraal", r"femoropatellair",
        r"επιγονατιδ", r"μηροεπιγονατιδ", r"τροχιλ",
        r"пател", r"феморопател", r"тролх",
        r"anterior compartment", r"compartimento anterior", r"prednj[^ ]* odjeljk",
    ),
}

# Achados autodeclarados: o próprio termo é o achado.
DIRECT = {
    "Effusion": _rx(
        r"\beffusion", r"joint fluid", r"intra ?articular fluid", r"\bhydrops\b",
        r"derrame articular", r"\bderrame\b", r"liquido articular",
        r"epanchement",
        r"gewrichtsvocht", r"\bvocht\b", r"\bhydrops\b", r"gewrichtseffusie",
        r"gelenkerguss", r"\berguss\b", r"gelenksergu",
        # "diz eklemi ici sivi miktari ... artmis" e "eklem icerisinde yaygin sivi
        # artisi" ambas ocorrem; o substantivo recebe um sufixo possessivo, então
        # `eklem ` sozinho falha. Casar o radical mais qualquer sufixo.
        r"eklem\w* ic\w* sivi", r"efuzyon", r"eklem sivisi",
        r"sivi (miktari|artisi|birikimi)", r"sivi artis", r"\bsivi\b[^.]{0,25}artmis",
        r"\bizljev", r"\bizliv", r"zglobn[^ ]* tekucin", r"\bhidrops\b",
        r"αρθρικ[^ ]* υγρ", r"υγρου ενδαρθρικα", r"ενδαρθρικ[^ ]* υγρ", r"ποσοτητα υγρου",
        r"ενδαρθρικ", r"αρθρικη συλλογη", r"υγρο στην αρθρωση", r"υγρου στην αρθρωση",
        r"ставен излив", r"излив", r"ставна течност", r"синовиална течност",
    ),
    "Synovitis": _rx(
        r"synovit", r"sinovit", r"synovial (thickening|proliferation|hypertroph)",
        r"synovitis", r"synoviale? (verdikking|proliferatie)",
        r"synovialitis", r"synovialis(verdickung|proliferation)",
        r"sinovijalitis", r"sinovitis", r"zadebljanje sinovij",
        r"υμενιτιδα", r"συνοβιτιδα", r"υμενικ[^ ]* υπερτροφ", r"αρθρικου υμεν",
        r"синовит", r"синовиал[^ ]* (задебел|пролифер)",
        r"verdikkingen van (het )?synovium", r"pannus",
    ),
    "Baker's": _rx(
        r"baker", r"popliteal cyst", r"quiste popliteo", r"quistes popliteos",
        r"kyste poplite", r"popliteale? cyst", r"poplitealzyste", r"bakerzyste",
        r"popliteal kist", r"\bbakerova\b", r"poplitealn[^ ]* cist",
        r"κυστη baker", r"πολυχωρη συνοβιακη κυστη", r"κυστη του baker",
        r"киста на бейкър", r"бейкърова киста", r"поплитеална киста",
        r"gastrocnemio ?semimembranos", r"gastrocnemius semimembranosus burs",
    ),
    "Contusion": _rx(
        r"\bcontusion", r"bone bruise", r"bone marrow (o?edema|contusion)",
        r"\bkontuz", r"medular bone o?edema", r"marrow o?edema",
        r"contusion osea", r"edema oseo", r"edema de medula osea",
        r"oedeme osseux", r"contusion osseuse",
        r"botcontusie", r"botoedeem", r"beenmergoedeem", r"botmergoedeem",
        r"knochenmarkodem", r"knochenodem", r"kontusion", r"bone bruise",
        r"kemik kontuzyonu", r"kemik iligi odemi", r"kemik odemi",
        r"kostani edem", r"edem kosti", r"kontuzij",
        r"οστεομυελικ[^ ]* οιδημα", r"οστικο οιδημα", r"μυελικο οιδημα",
        r"костномозъчен едем", r"костен едем", r"контузионен",
    ),
    "Fracture": _rx(
        r"\bfractur", r"\bfract\b",
        r"\bfractura", r"\bfracturas\b",
        r"\bfractuur", r"\bbreuk\b",
        r"\bfraktur", r"\bbruch\b",
        r"\bkirik\b", r"\bkirigi\b", r"\bkirik\b",
        r"\bfraktur", r"\bprijelom", r"impresijsk[^ ]* fraktur",
        r"καταγμα", r"καταγματ",
        r"фрактур", r"счупван", r"фисур",
        r"insufficiency fracture", r"stress fracture", r"avulsion fracture",
        r"subchondral fracture", r"subkondral kiri",
    ),
}

# Termos que parecem um achado mas não são o achado sendo pontuado.
DECOY = {
    # `no fracture` está deliberadamente ausente: um decoy pula a cláusula inteira, então
    # listá-lo aqui transformou a negação mais comum em inglês em silêncio, e o estudo
    # passava a puxar a cabeça de fratura com o peso de um laudo que nunca mencionou
    # fratura nenhuma. `microfractur` é um procedimento cirúrgico e `fracture risk` uma
    # previsão; os dois ficam.
    "Fracture": _rx(r"microfractur", r"\bfracture (risk|prophyla)"),
    "Baker's": _rx(r"meniscal cyst", r"quiste meniscal", r"ganglion"),
}

PAIRED = {"ACL", "MCL", "Medial Meniscus", "Lateral Meniscus"}
OA_TARGETS = {"Medial OA", "Lateral OA", "PF OA"}


In [ ]:
STEM_MENISCUS = _rx(r"menisc\w*", r"menisk\w*", r"μηνισκ\w*", r"мениск\w*")
STEM_CRUCIATE = _rx(r"cruciate", r"cruzado", r"croise", r"kruisband", r"kreuzband",
                    r"capraz bag\w*", r"krizn\w*", r"χιαστ\w*", r"кръстн\w*",
                    r"\bacl\b", r"\bpcl\b", r"\blca\b", r"\blcp\b", r"\bvkb\b",
                    r"\bhkb\b", r"\bocb\b", r"\bacb\b")
STEM_COLLATERAL = _rx(r"collateral\w*", r"colateral\w*", r"kollateral\w*",
                      r"collaterale\w*", r"kolateraln\w*", r"yan bag\w*",
                      r"πλαγι\w*", r"колатерал\w*", r"странич\w*",
                      r"innenband\w*", r"aussenband\w*", r"binnenband\w*",
                      r"\bmcl\b", r"\blcl\b", r"\blcm\b", r"\biyb\b")

SIDE_MEDIAL = _rx(r"\bmedial\w*", r"\bmedyal\w*", r"\bmedijaln\w*", r"\bmediaal\w*",
                  r"\bmediale\w*", r"\bintern[oa]\w*", r"\binterne\w*", r"\binnen\w*",
                  r"\bic\b", r"\bunutarnj\w*", r"\bεσω\w*", r"\bεσωτερικ\w*",
                  r"\bмедиал\w*", r"\bвътреш\w*", r"\btibial collateral\b",
                  r"\bbinnen\w*", r"\bmediaal\b")
SIDE_LATERAL = _rx(r"\blateral\w*", r"\bextern[oa]\w*", r"\bexterne\w*", r"\bdis\b",
                   r"\blateraln\w*", r"\baussen\w*", r"\bbuiten\w*", r"\bεξω\w*",
                   r"\bεξωτερικ\w*", r"\bлатерал\w*", r"\bвъншн\w*",
                   r"\bfibular collateral\b", r"\bvanjsk\w*")
SIDE_ANTERIOR = _rx(r"\banterior\w*", r"\bant\b", r"\bon\b", r"\bprednj\w*",
                    r"\bvorder\w*", r"\bvoorste\b", r"\bπροσθι\w*", r"\bпредн\w*",
                    r"\banteriyor\w*", r"\bavant\b", r"\bant[eé]rieur\w*")

# O contrário de SIDE_ANTERIOR, necessário só pra impedir que uma pista de cruzado cega
# a lado dispare no ligamento posterior. Nunca é usado pra afirmar um alvo - não há alvo
# de LCP - então é deliberadamente estreito: `posterior horn` é uma das frases mais
# comuns num laudo de joelho e não deve ser lida como um qualificador de cruzado, por
# isso a guarda abaixo testa proximidade ao radical do cruzado em vez de presença na
# cláusula.
SIDE_POSTERIOR = _rx(r"\bposterior\w*", r"\bpost[eé]rieur\w*", r"\bposteriore\w*",
                     r"\bhinter\w*", r"\bachterste\b", r"\barka\b", r"\bstraznj\w*",
                     r"\bzadnj\w*", r"\bοπισθι\w*", r"\bзадн\w*", r"\bpostero\w*")

# Fracture é o alvo cujo radical mais varia pelo corpus.
STEM_FRACTURE = _rx(r"fractur\w*", r"fraktur\w*", r"fractuur\w*", r"\bfract\b",
                    r"kiri[kgğ]\w*", r"prijelom\w*", r"lom kosti", r"\bbreuk\w*",
                    r"\bbruch\w*", r"καταγμα\w*", r"καταγματ\w*", r"фрактур\w*",
                    # NÃO um `fissur\w*` isolado: "fisuras condrales" e "full thickness
                    # fissures in the articular cartilage" descrevem cartilagem, não osso.
                    # O radical tem que estar ancorado a uma palavra de osso pra
                    # significar fratura.
                    r"счупван\w*", r"fisur\w* (osea|oseas|kost)", r"fissur\w* kost")

STEM_OA_COMPARTMENT = _rx(r"compartment\w*", r"compartimento\w*", r"compartiment\w*",
                          r"kompartman\w*", r"kompartiment\w*", r"odjelj\w*",
                          r"διαμερισμα\w*", r"компартм\w*", r"\bотдел\w*",
                          r"femorotibial\w*", r"femorotibiaal\w*", r"tibiofemoral\w*",
                          r"femoro tibial\w*", r"κνημιαι\w*", r"μηριαι\w*",
                          r"femoral condyl\w*", r"tibial plateau\w*",
                          r"condilo femoral", r"platillo tibial", r"tibiaplateau\w*",
                          r"femurkondyl\w*", r"femoralne? kondil\w*",
                          r"tibijaln\w* plato", r"femoral kondil\w*",
                          r"tibia plato", r"tibyal plato")


def _distance(clause: str, stem_rx: re.Pattern, qual_rx: re.Pattern, window: int = 55):
    """Caracteres do radical mais próximo até o qualificador mais próximo, ou None se
    nenhum estiver perto.

    Janelas de caracteres em vez de janelas de tokens, porque a ordem das palavras
    difere: o inglês põe o lado antes do substantivo, o grego e o búlgaro geralmente
    depois, e o turco o anexa como um adjetivo separado que precede.

    Uma distância em vez de um sim. Presença basta pra decidir que um qualificador se
    aplica a uma estrutura, mas não basta pra decidir qual de dois qualificadores se
    aplica: um laudo de joelho diz "anterior horn" e "posterior horn" o tempo todo, então
    qualquer janela larga o bastante pra pegar uma palavra de lado de verdade também pega
    uma não relacionada, e duas regras que ambas respondem "sim" não podem ser
    distinguidas. Comparar a que distância cada uma está pode.
    """
    best = None
    for m in stem_rx.finditer(clause):
        lo = max(0, m.start() - window)
        hi = min(len(clause), m.end() + window)
        for q in qual_rx.finditer(clause[lo:hi]):
            qs, qe = lo + q.start(), lo + q.end()
            d = 0 if qs < m.end() and qe > m.start() else \
                min(abs(m.start() - qe), abs(qs - m.end()))
            best = d if best is None else min(best, d)
    return best


def _near(clause: str, stem_rx: re.Pattern, qual_rx: re.Pattern, window: int = 55):
    """True se um match de radical tem um qualificador dentro de `window` caracteres,
    em qualquer direção."""
    return _distance(clause, stem_rx, qual_rx, window) is not None
    return False


# conceito -> pares (radical, lado) usados além dos léxicos de frases acima
STEM_RULES = {
    "ACL": (STEM_CRUCIATE, SIDE_ANTERIOR),
    "MCL": (STEM_COLLATERAL, SIDE_MEDIAL),
    "Medial Meniscus": (STEM_MENISCUS, SIDE_MEDIAL),
    "Lateral Meniscus": (STEM_MENISCUS, SIDE_LATERAL),
    "Medial OA": (STEM_OA_COMPARTMENT, SIDE_MEDIAL),
    "Lateral OA": (STEM_OA_COMPARTMENT, SIDE_LATERAL),
}


In [ ]:
SEV_LOW = _rx(
    r"\bsmall\b", r"\bminimal\b", r"\btrace\b", r"\bmild\b", r"\bslight\b",
    r"\btiny\b", r"\bscant\b", r"\bmimimal\b", r"\bdiscrete\b", r"\bfocal\b",
    r"\bleve\b", r"\bminim", r"\bpeque", r"\bligero\b", r"\bescaso\b", r"\bdiscreto\b",
    r"\bhafif\b", r"\bminimal\b", r"\baz miktarda\b", r"\bsilik\b",
    r"\bmanja\b", r"\bmanji\b", r"\bblago\b", r"\bdiskretn", r"\bmalo\b",
    r"\bgering", r"\bdiskret", r"\bkleine?r?\b", r"\bwenig\b", r"\bzarte?\b",
    r"\bbeperkte?\b", r"\bgeringe\b", r"\bweinig\b", r"\blichte?\b",
    r"\bηπι", r"\bμικρ", r"\bελαχιστ",
    r"\bминимал", r"\bлек", r"\bмалк", r"\bнеголям",
)

SEV_HIGH = _rx(
    r"\blarge\b", r"\bmarked\b", r"\bmassive\b", r"\bsevere\b", r"\bextensive\b",
    r"\bmoderate\b", r"\bgross\b", r"\bsignificant\b", r"\babundant\b", r"\btense\b",
    r"\bmoderad", r"\bimportante\b", r"\bsevera?\b", r"\bmarcad", r"\bcuantios",
    r"\bbelirgin\b", r"\byaygin\b", r"\bileri\b", r"\bciddi\b", r"\bbol\b",
    r"\bopsezan\b", r"\bveliki\b", r"\bizrazit", r"\bznacajn", r"\bumjeren",
    r"\bausgepragt", r"\bdeutlich", r"\bmassiv", r"\bmassig", r"\bgross",
    r"\buitgebreid", r"\bgevorderd", r"\bveel\b", r"\bmatige?\b",
    r"\bμετρι", r"\bμεγαλ", r"\bεκτεταμεν", r"\bευμεγεθ", r"\bσοβαρ",
    r"\bголям", r"\bизразен", r"\bзначим", r"\bумерен", r"\bобилен",
)

# OA frequentemente é afirmada pra articulação inteira em vez de por compartimento
# ("tricompartmental osteoarthritis", "gonarthrose", "incipient OA of all three
# compartments"). Essas afirmações são evidência pros três alvos de OA.
GLOBAL_OA = _rx(
    r"tri ?compartment", r"all three compartment", r"global(ised)? (oa|osteoarthrit)",
    r"\bgonarthros", r"\bgonartros", r"\bgonarthrose", r"\bgonartrose",
    r"osteoarthritis of the knee", r"artrosis (de |)(la )?rodilla", r"knee osteoarthrit",
    r"\bdiz osteoartrit", r"\bgonartroz", r"artroza koljena",
    r"οστεοαρθριτιδα", r"αρθριτιδα του γονατος",
    r"артроза на колянната", r"гонартроз",
    r"degenerative joint disease", r"\bdjd\b",
)

# Um "bone marrow oedema" isolado não é uma contusão quando está sob um defeito de
# cartilagem: edema subcondral abaixo de um compartimento desgastado é sinal
# degenerativo reativo, e lê-lo como contusão transforma todo joelho osteoartrítico num
# caso de trauma.
DEGENERATIVE_MARROW = _rx(
    r"subchondral", r"subcondral", r"subkondral", r"supkondraln", r"subchondraln",
    r"υποχονδρι", r"субхондрал", r"subchondrale?",
    r"\bcyst", r"\bquist", r"\bzyste\b", r"\bcistic", r"reactive", r"reactivo",
)

TRAUMA = _rx(
    r"\bbruise\b", r"\bcontusion", r"\bkontuz", r"\bcontusion osea\b",
    r"\btrauma", r"\bimpaction\b", r"\bpivot shift\b", r"\bkissing\b",
    r"\bacute\b", r"\bagudo\b", r"\bakut", r"\bpivot kaymasi\b",
    r"\bcontusion osseuse\b", r"\bbone bruise\b", r"\bbotcontusie\b",
    r"\bконтузион", r"\bμωλωπ", r"\bkontuzij",
)


In [ ]:
def _polarity(clause: str, anchor_end: int) -> str:
    """Classifica uma cláusula como positiva, negativa ou incerta pra um termo casado.

    O escopo é a cláusula inteira. A segmentação de cláusulas já mantém as afirmações
    curtas, e uma janela em caracteres delimita mal entre idiomas com ordens de palavra
    diferentes - o turco põe seu negador no fim da frase, o inglês no começo.
    """
    if UNCERTAIN.search(clause):
        return "uncertain"
    if NEGATION.search(clause):
        return "negative"
    if NORMALITY.search(clause):
        # "meniscus normal" nega; "normal ... but tear" não nega.
        if TEAR.search(clause) or re.search(r"\bgrade [34]\b", clause):
            return "positive"
        return "negative"
    return "positive"


class _Matcher:
    """Léxico de frases primeiro, proximidade radical+lado como fallback.

    Expõe `.search` pra encaixar no mesmo lugar que um pattern compilado.
    """

    def __init__(self, phrase_rx, stem=None, side=None, window=55, contrary=None):
        self.phrase_rx = phrase_rx
        self.stem = stem
        self.side = side
        self.window = window
        self.contrary = contrary

    def search(self, clause):
        m = self.phrase_rx.search(clause)
        if m is not None and not self._wrong_side(clause):
            return m
        if self.stem is not None and _near(clause, self.stem, self.side, self.window):
            return self.stem.search(clause)
        return None

    def _wrong_side(self, clause):
        """True quando a cláusula nomeia o outro membro do par dessa estrutura.

        Algumas pistas no léxico são cegas a lado por design: o grego separa o adjetivo
        do substantivo ("cruciate and collateral ligaments"), então o radical nu do
        adjetivo tem que ficar sozinho ou a cláusula se perde. Esse radical então também
        casa com o cruzado posterior e o colateral lateral, nenhum dos dois um alvo aqui,
        e um positivo supera todo negativo no scorer - então uma cláusula sobre o LCP era
        suficiente pra sobrepor um explícito "o LCA está normal".

        O teste é proximidade ao radical da própria estrutura, não presença na cláusula.
        "Posterior horn of the medial meniscus" aparece numa grande fração dos laudos de
        joelho e não diz nada sobre um cruzado; só um qualificador ao lado da palavra do
        ligamento é um. Uma cláusula nomeando os dois lados mantém o match, porque ela de
        fato menciona esse alvo.
        """
        if self.contrary is None or self.stem is None:
            return False
        other = _distance(clause, self.stem, self.contrary, self.window)
        if other is None:
            return False
        own = _distance(clause, self.stem, self.side, self.window)
        # Não "o outro lado é mencionado" mas "é ele o mais próximo dos dois". Uma
        # cláusula que lê "tear of the posterior horn of the medial meniscus; the
        # cruciate ligaments are intact" menciona posterior, e sob um teste de presença
        # isso bastava pra suprimir a pista de cruzado - o oposto da intenção, já que a
        # cláusula de fato descreve os ligamentos. Empates ficam com o match mantido: uma
        # cláusula nomeando os dois lados de fato menciona este aqui.
        return own is None or other < own


# Qual pista, se estiver ao lado do radical da estrutura, significa que a cláusula é
# sobre o outro membro do par. Só as duas estruturas com pista cega a lado precisam de
# uma.
CONTRARY = {"ACL": SIDE_POSTERIOR, "MCL": SIDE_LATERAL}

ANAT_MATCH = {
    tgt: _Matcher(ANAT[tgt], *STEM_RULES[tgt], contrary=CONTRARY.get(tgt))
    for tgt in PAIRED
}
COMPARTMENT_MATCH = {
    "Medial OA": _Matcher(COMPARTMENT["Medial OA"], *STEM_RULES["Medial OA"]),
    "Lateral OA": _Matcher(COMPARTMENT["Lateral OA"], *STEM_RULES["Lateral OA"]),
    "PF OA": _Matcher(COMPARTMENT["PF OA"]),
}
DIRECT_MATCH = {
    tgt: _Matcher(_rx(rx.pattern, STEM_FRACTURE.pattern) if tgt == "Fracture" else rx)
    for tgt, rx in DIRECT.items()
}


def _severity(clause: str) -> float:
    """Pesa uma menção positiva pelo quão enfática a frase é.

    Ordenado, não calibrado. Uma "moderate effusion" tem que superar uma "trace
    effusion" e ambas têm que superar o silêncio; os números absolutos não importam pra
    AUC.
    """
    high = SEV_HIGH.search(clause) is not None
    low = SEV_LOW.search(clause) is not None
    if high and not low:
        return 1.0
    if low and not high:
        return 0.45
    return 0.75                       # menção sem qualificador


def _score_clauses(cls, anat_rx, path_rx=None, decoy_rx=None, context_penalty=None,
                   context_bonus=None):
    """Acumula evidência graduada pelas cláusulas para um alvo.

    Retorna (score, confidence, n_pos, n_neg). Positivos são graduados por severidade e
    por regexes de contexto opcionais; negativos só importam quando nenhum positivo foi
    encontrado, porque laudos afirmam normalidade pra toda estrutura que checam.
    """
    n_pos = n_neg = n_unc = 0
    best = 0.0
    for c in cls:
        m = anat_rx.search(c)
        if not m:
            continue
        if decoy_rx is not None and decoy_rx.search(c):
            continue
        if path_rx is not None and not path_rx.search(c):
            if NORMALITY.search(c) and not NEGATION.search(c):
                n_neg += 1
            continue
        pol = _polarity(c, m.end())
        if pol == "positive":
            n_pos += 1
            w = _severity(c)
            if context_penalty is not None and context_penalty.search(c):
                w *= 0.45
            if context_bonus is not None and context_bonus.search(c):
                w = min(1.0, w * 1.35)
            best = max(best, w)
        elif pol == "negative":
            n_neg += 1
        else:
            n_unc += 1
            best = max(best, 0.30)

    if n_pos or n_unc:
        # 0.52 .. 0.95, ordenado pela menção mais forte, com leve ajuste por repetição.
        score = min(0.95, 0.50 + 0.42 * best + 0.03 * min(n_pos, 3))
        conf = min(1.0, 0.55 + 0.15 * n_pos)
    elif n_neg:
        score = max(0.04, 0.20 - 0.04 * n_neg)
        conf = min(0.9, 0.45 + 0.12 * n_neg)
    else:
        score, conf = 0.28, 0.05          # silêncio fica acima de negativo-afirmado
    return score, conf, n_pos, n_neg


def extract(report: str) -> dict:
    """Extrai doze pares (score, confidence) de um laudo."""
    cls = clauses(report)
    out = {}
    path_paired = _rx(TEAR.pattern, DEGEN.pattern, INJURY.pattern)

    for tgt in TARGETS:
        if tgt in PAIRED:
            s, c, npos, nneg = _score_clauses(cls, ANAT_MATCH[tgt], path_paired)
        elif tgt in OA_TARGETS:
            s, c, npos, nneg = _score_clauses(cls, COMPARTMENT_MATCH[tgt], OA_EVIDENCE)
        elif tgt == "Contusion":
            # Edema subcondral reativo sob um defeito de cartilagem é osteoartrite, não
            # contusão. Linguagem explícita de trauma empurra na direção contrária.
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt),
                                              context_penalty=DEGENERATIVE_MARROW,
                                              context_bonus=TRAUMA)
        else:
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt))
        out[tgt] = s
        out[tgt + "__conf"] = c
        out[tgt + "__npos"] = npos
        out[tgt + "__nneg"] = nneg

    # --- correções entre alvos ---------------------------------------------- #
    # Uma afirmação de osteoartrite pra articulação inteira é evidência pra todo
    # compartimento que não foi avaliado separadamente. Sem isso, "incipient OA of all
    # three compartments" pontuaria zero nos três alvos de OA.
    g_hits = [c for c in cls if GLOBAL_OA.search(c) and _polarity(c, 0) == "positive"]
    if g_hits:
        gscore = 0.50 + 0.42 * max(_severity(c) for c in g_hits)
        for tgt in OA_TARGETS:
            if out[tgt + "__npos"] == 0 and out[tgt + "__nneg"] == 0:
                out[tgt] = max(out[tgt], gscore * 0.92)
                out[tgt + "__conf"] = max(out[tgt + "__conf"], 0.4)

    # Sinovite frequentemente é visível na imagem e ausente do texto, então silêncio é
    # evidência fraca de ausência aqui de um jeito que não é pra outros achados. Efusão é
    # seu proxy textual mais confiável - os dois compartilham um mecanismo - então uma
    # sinovite silenciosa herda uma fração da evidência de efusão em vez de cair ao piso.
    if out["Synovitis__npos"] == 0 and out["Synovitis__nneg"] == 0:
        out["Synovitis"] = max(out["Synovitis"], 0.28 + 0.45 * (out["Effusion"] - 0.28))

    return out


## 3. Lendo a aquisição

`train_series.csv` descreve cada série com um plano anatômico e duas flags binárias,
`Fluid_Sensitive` e `Fat_Suppression`. Os nomes denotam duas propriedades fisicamente
independentes — e é essa independência que as colunas entregues não têm: entre as séries
de treino as duas concordam em toda linha, então, do jeito que vêm, elas carregam um
único eixo em vez de dois. Esse é o primeiro motivo pra recuperar as duas a partir do
header.

*Sensibilidade a fluido* é uma propriedade da **ponderação de contraste**, definida pelo
tempo de repetição $T_R$ e pelo tempo de eco $T_E$:

$$
\text{ponderação} \;=\;
\begin{cases}
T_1 & T_R \lesssim 800\ \text{ms}\\
T_2 & T_R \gtrsim 800\ \text{ms},\ T_E \gtrsim 60\ \text{ms}\\
\text{PD} & T_R \gtrsim 800\ \text{ms},\ T_E \lesssim 60\ \text{ms}
\end{cases}
$$

Fluido é claro em $T_2$, intermediário em densidade de prótons, escuro em $T_1$.
Gradiente-eco quebra a regra — seu $T_R$ é curto por design — então é resolvido primeiro
por `ScanningSequence`. *Supressão de gordura* é uma **preparação** aplicada por cima de
qualquer ponderação, e é o que torna o edema de medula óssea evidente. As duas são lidas
de `SeriesDescription`, `SequenceName` e `ScanOptions` onde o protocolo as nomeia, e de
$T_R$ e $T_E$ onde não nomeia.

### Quais sequências mostrar ao modelo

Um joelho é lido em três planos porque as estruturas correm em direções diferentes:
ligamentos cruzados obliquamente, melhor vistos sagitalmente; ligamentos colaterais e o
corpo do menisco coronalmente; cartilagem patelar e retináculos axialmente. Cruzar plano
com os dois eixos de aquisição dá os slots abaixo, escolhidos pra que cada um dos doze
achados tenha pelo menos uma sequência que o mostre bem.

| slot | plano | ponderação | supressão de gordura | o que carrega |
|---|---|---|---|---|
| `SAG_FLUID_FS` | sagital | PD / T2 | sim | rupturas meniscais, edema de medula, efusão |
| `COR_FLUID_FS` | coronal | PD / T2 | sim | ligamentos colaterais, corpo do menisco, edema |
| `AX_FLUID_FS` | axial | PD / T2 | sim | articulação patelofemoral, sinóvia, efusão |
| `SAG_FLUID_NOFS` | sagital | PD / T2 | não | morfologia meniscal com alto contraste-ruído |
| `COR_T1` | coronal | T1 | não | arquitetura da medula, contorno de cartilagem e osso |
| `SAG_T1` | sagital | T1 | não | anatomia, alteração crônica |

Um estudo raramente tem todos os seis; uma máscara de presença por slot carrega as
ausências para a cabeça (head), que o §6 usa.


In [ ]:
from __future__ import annotations

import os

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "4")

import gc
import hashlib
import json
import re
import time
import traceback
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F

# O extrator de labels é definido nas células acima quando isso roda como notebook.
# Como script puro, ele é importado do código-fonte do pacote, então os dois caminhos
# compartilham uma única definição em vez de manter uma cópia cada.

T0 = time.time()
SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]


# O corte central tem que ser menor que o menor campo de visão do corpus, senão não faz
# nada silenciosamente. Medido em cada série de treino, o campo de visão adquirido
# (Rows x PixelSpacing) tem mediana de 160 mm e varia de 70 a 320: um corte de 160 mm é
# maior que a imagem em 60% das séries e é pulado em todas elas, o que deixa a escala
# física delas não normalizada. 130 mm fica abaixo do campo de visão de 99.6% das séries
# e ainda contém a articulação.
CROP_MM = 130.0

# Resolução do cache. Tudo a jusante pode reduzir a partir disso, então é definido pela
# configuração mais exigente, não pela padrão.
CACHE_IMG = 336
GROUP = 3                  # slices por input do encoder, empilhados como os três canais
N_GROUP_MAX = 1
CACHE_FRACTION = 0.45      # fração da memória livre que o cache de pixels pode usar
CACHE_BUDGET_MAX_GB = 24.0 # teto rígido independente do que a máquina reportar
CACHE_BUDGET_GB = 12.0     # só o fallback, pra uma máquina sem /proc/meminfo
TEST_SHARE = 0.30          # piso pro corpus de teste relativo ao de treino, já que o
                           # split de teste visível é um stub e o que é pontuado não é
HDR_THREADS = 16
PIX_THREADS = 12
ORDER_THREADS = 32         # ordenar slices é limitado por latência no mount, não por CPU
# Teto pra passada de ordenação. Precisa ser um teto porque a passada é centenas de
# milhares de leituras pequenas num mount de rede, então sua duração é uma propriedade do
# mount, não do trabalho, e varia entre execuções que fazem a mesma leitura. Não pode ser
# um teto apertado: desistir deixa aquelas séries na ordem de arquivo, que não se
# correlaciona com anatomia, e essa degradação é silenciosa. Então o teto fica bem acima
# do que a passada normalmente precisa: seu propósito é impedir que a passada consuma a
# execução inteira num mount lento, não cortar o caso comum, e um teto apertado o
# bastante pra vincular num dia normal trocaria uma degradação silenciosa por uma
# economia que a execução não precisa.
ORDER_BUDGET_S = 5400

# Resolução é o eixo sob teste. Um traço de largura d mm sobrevive ao reamostragem só se
# o passo de pixel for no máximo d/2, e o passo aqui é definido pelo corte acima, não
# pelo campo de visão adquirido: CROP_MM / P. A 224 px isso é 0.58 mm, acima dos 0.5 mm
# que uma ruptura de 1 mm precisa; a 336 px é 0.39 mm e passa no critério. As duas
# configurações leem o mesmo cache, então a comparação isola o resize.
RUNS = [
    {"name": "r224", "img": 224},
    {"name": "r336", "img": 336},
]

EPOCHS = 10
BATCH_STUDIES = 8          # um estudo é uma bag de até N_SLOT imagens de slot
AUG_ROT_DEG = 8.0          # jitter rígido; ver augment() pra saber por que nenhum flip é usado
AUG_SCALE = 0.08
AUG_SHIFT = 0.05
AUG_INTENSITY = 0.10
LAT_MIN_OFFSET_MM = 20.0   # dentro disso o lado não é legível a partir da geometria; ver
                           # side_from_geometry()
SLICE_BAND = (0.20, 0.80)  # fração da pilha ordenada que read_slot amostra

# --- O que uma slice É, em oposição a quantas delas existem --------------- #
#
# Um member é uma função dos pixels em que foi ajustado, e img/crop_mm/slices/band não
# determinam esses pixels por si só. Quatro outras decisões determinam, nenhuma delas
# visível em nenhum shape:
#
#   order          qual slice é a próxima ao longo da pilha
#   lat            quais joelhos são espelhados, e com base em que evidência
#   slot_fallback  se um slot T1 pode ser preenchido por uma série que não é T1
#   decode_fill    o que substitui uma slice que não decodificaria
#
# `native` é a leitura derivada nas seções abaixo. `legacy` é a leitura sob a qual um
# member importado foi ajustado. Um member lido sob a leitura errada carrega com todo
# shape batendo, roda, e escreve uma submissão plausível calculada a partir da imagem
# errada - então a escolha viaja com o member e é parte da chave que decide quais members
# podem compartilhar um decode. As regras legacy são reproduzidas em vez de corrigidas:
# corrigi-las entregaria a esse member pixels que seus pesos nunca viram.
RULES_NATIVE = {"order": "normal", "lat": "centre",
                "slot_fallback": False, "decode_fill": "nearest"}
RULES_LEGACY = {"order": "dominant_axis", "lat": "corner_x",
                "slot_fallback": True, "decode_fill": "zero"}
RULES = dict(RULES_NATIVE)
LEGACY_LAT_OFFSET_MM = 5.0   # a zona morta com a qual a regra legacy de lateralidade foi ajustada

LR_HEAD = 1e-3
LR_BACKBONE = 8e-6         # o encoder é adaptado, não retreinado do zero
UNFREEZE_LAST = 6          # blocos do transformer treináveis, a partir do fim da saída
WEIGHT_DECAY = 0.02
EVAL_BATCH = 8
TIME_BUDGET = 8.0 * 3600

# Seis slots: três planos cruzados com os eixos de aquisição. A série fluid-sensitive com
# supressão de gordura existe pra quase todo estudo; as séries T1 e fluid-sensitive sem
# supressão são mais raras, que é pra isso que serve a máscara de presença.
SLOTS_RECOVERED = [
    ("SAG_FLUID_FS", "Sagittal", True, True),
    ("COR_FLUID_FS", "Coronal", True, True),
    ("AX_FLUID_FS", "Axial", True, True),
    ("SAG_FLUID_NOFS", "Sagittal", True, False),
    ("COR_T1", "Coronal", False, False),
    ("SAG_T1", "Sagittal", False, False),
]

# A alternativa: plano x o único eixo que as flags entregues carregam, ignorando a
# ponderação recuperada. Mantido como uma opção pra que a escolha de definição de slot
# possa variar enquanto tudo o mais fica fixo. Sob esse esquema, um slot `Struct` mistura
# séries T1 com séries PD/T2 sem supressão de gordura, que carregam contraste de tecido
# bem diferente.
SLOTS_PUBLIC = [
    ("SAG_FLUID", "Sagittal", None, True),
    ("COR_FLUID", "Coronal", None, True),
    ("AX_FLUID", "Axial", None, True),
    ("SAG_STRUCT", "Sagittal", None, False),
    ("COR_STRUCT", "Coronal", None, False),
    ("AX_STRUCT", "Axial", None, False),
]

SLOT_SCHEME = os.environ.get("SLOT_SCHEME", "recovered")
SLOTS = SLOTS_PUBLIC if SLOT_SCHEME == "public" else SLOTS_RECOVERED
N_SLOT = len(SLOTS)

# Quantas partes de largura 384 compõem a feature por slot. O encoder emite um vetor por
# token; uma feature de slot é um resumo fixo dessa grade, e o resumo com o qual um
# member importado foi ajustado carrega uma terceira parte.
POOL_PARTS = {"cls_mean": 2, "cls_mean_focal": 3}

# Pra quais slots a atenção de um member importado se inclina, por diagnóstico. Índices
# são em SLOTS. Isso é uma tabela fixa em vez de um parâmetro aprendido, então é parte da
# definição daquele member e tem que ser reproduzida exatamente pra seus pesos fazerem
# sentido.
SLOT_PRIOR_TABLE = {
    "ACL": (0, 3, 5), "MCL": (1, 4),
    "Medial Meniscus": (0, 1, 3, 4), "Lateral Meniscus": (0, 1, 3, 4),
    "Medial OA": (1, 4, 5), "Lateral OA": (1, 4, 5),
    "PF OA": (0, 2, 5), "Effusion": (0, 2), "Synovitis": (0, 2),
    "Baker's": (0,), "Contusion": (0, 1, 2), "Fracture": (0, 1, 2, 4, 5),
}
SLOT_PRIOR_STRENGTH = 0.55

FATSAT_OPTS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}
_SEP = re.compile(r"[_\-.]")
_FATSAT_RX = re.compile(r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|"
                        r"water excit|\btirm\b|\bsting\b|\bfatsup\b")
_T1_RX = re.compile(r"\bt1\b|\bt1w\b")
_T2_RX = re.compile(r"\bt2\b|\bt2w\b")
_PD_RX = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")


In [ ]:
def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)


def find_root():
    for c in [Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
              Path("/kaggle/input/rsna-knee-abnormality-detection"),
              Path("data"), Path(".")]:
        if (c / "test.csv").is_file() and (c / "test_series").is_dir():
            return c
    # último recurso: varredura de dois níveis, porque o mount está aninhado um nível
    # mais fundo que o normal
    base = Path("/kaggle/input")
    if base.is_dir():
        for depth1 in sorted(p for p in base.iterdir() if p.is_dir()):
            for cand in [depth1] + sorted(p for p in depth1.iterdir() if p.is_dir()):
                if (cand / "test.csv").is_file():
                    return cand
    raise FileNotFoundError(
        f"competition mount not found (cwd {Path.cwd()}); expected a directory holding "
        f"test.csv and test_series/")


def find_dinov2(variant="small"):
    """Localiza um diretório de checkpoint DINOv2 montado pelo nome da variante."""
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    hits = []
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
        if "config.json" in files and "dinov2" in root.lower():
            hits.append(Path(root))
    for h in hits:
        if variant in str(h).lower():
            return h
    return hits[0] if hits else None


LABEL_COLS = TARGETS + [t + "__conf" for t in TARGETS]


class LabelSourceError(RuntimeError):
    """Lançada quando os labels não vieram de onde esta execução pretendia.

    Toda outra falha neste arquivo é melhor sobrevivida do que reportada: uma execução
    que morre depois que o cache é construído já gastou a metade cara e não pontua nada,
    então a guarda em torno de `main` a engole e deixa o arquivo de benchmark pra trás.
    Esta é a exceção. Treinar com os labels mais fracos não parece uma falha - completa,
    escreve uma submissão plausível, e difere só numa linha de log - então tem que parar
    a execução em vez de ser absorvida por uma guarda desenhada pra crashes.
    """


def find_label_table():
    """Localiza uma tabela montada de labels de laudo já lidos, se alguma estiver anexada.

    O léxico transforma um laudo em labels casando morfologia, e seu modo de falha é
    silêncio: numa formulação que ele não carrega, ele não emite opinião nenhuma em vez
    de uma errada. Silêncio é mensurável sem nenhum gabarito - pra cada par (laudo,
    achado), algo casou? - e essa medida diz que as falhas se concentram em idiomas
    específicos em vez de espalhadas igualmente, em achados que um laudo de joelho quase
    sempre comenta.

    Enumerar morfologia pra nove idiomas é o instrumento errado pra isso. Ler a frase é o
    certo, e um modelo de linguagem a lê. Contra os estudos anotados a diferença é grande
    e unilateral, então quando essa tabela está montada ela é preferida; quando não está,
    o léxico roda e o pipeline continua inalterado. Os dois caminhos produzem as mesmas
    colunas, então nada a jusante sabe qual deles as forneceu.
    """
    base = Path("/kaggle/input")
    cands = []
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
            cands += [Path(root) / f for f in files if f.startswith("report_labels")
                      and f.endswith(".csv")]
    cands += [p for p in (Path("data/derived/report_labels_v2.csv"),) if p.is_file()]
    for c in cands:
        try:
            head = pd.read_csv(c, nrows=1)
        except Exception:
            continue
        if "StudyInstanceUID" in head.columns and all(t in head.columns for t in TARGETS):
            return c
    return None


def label_mount_attached():
    """True quando um diretório de input foi anexado com a intenção de carregar uma
    tabela de labels.

    O fallback abaixo é deliberado e precisa ficar silencioso pra uma execução sem
    nenhuma tabela anexada, porque esse é o caso comum pra quem estiver lendo este
    notebook. Ele não pode ficar silencioso pro outro caso: uma tabela foi anexada e não
    pôde ser usada. Os dois casos são indistinguíveis só pelos labels - ambos terminam no
    léxico - então são separados aqui por se o mount sequer existe.
    """
    base = Path("/kaggle/input")
    if not base.is_dir():
        return False
    return any("label" in p.name.lower() for p in base.iterdir() if p.is_dir())


def read_labels(train_df):
    """Labels pra todo estudo de treino, de uma tabela montada ou do léxico.

    Estudos que a tabela montada não cobre recorrem ao léxico em vez de serem
    descartados, então uma tabela parcial degrada cobertura em vez de perder linhas.
    """
    n = len(train_df)
    lab = pd.DataFrame([extract(r) for r in train_df["Report"].fillna("")])
    lab["StudyInstanceUID"] = train_df["StudyInstanceUID"].values
    lab = lab.set_index("StudyInstanceUID")

    src = find_label_table()
    if src is None:
        if label_mount_attached():
            raise LabelSourceError(
                "LABEL SOURCE: a label dataset is mounted but no usable table was found "
                "in it. Falling back to the lexicon here would train on the weaker "
                "labels and say so only in a log line, so the run stops instead.")
        log(f"LABEL SOURCE: lexicon, {n} studies (no table mounted)")
        return lab

    tab = pd.read_csv(src).set_index("StudyInstanceUID")
    missing = [c for c in LABEL_COLS if c not in tab.columns]
    if missing:
        raise LabelSourceError(
            f"LABEL SOURCE: {src} is missing {len(missing)} expected columns "
            f"(first: {missing[0]!r}). Refusing to fall back silently.")
    hit = lab.index.intersection(tab.index)
    if not len(hit):
        raise LabelSourceError(
            f"LABEL SOURCE: {src} shares no StudyInstanceUID with train.csv.")
    log(f"LABEL SOURCE: {src.name} covers {len(hit)} of {n} studies, "
        f"lexicon for the remaining {n - len(hit)}")
    lab.loc[hit, LABEL_COLS] = tab.loc[hit, LABEL_COLS].values
    return lab


ROOT = find_root()
log(f"input root: {ROOT}")


IMG = CACHE_IMG            # mantido como o nome que o leitor de pixels e o cache usam


def available_gb():
    """Memória que esta máquina realmente vai ceder, lida em vez de presumida.

    Um teto fixo é um palpite sobre uma máquina em que o autor não está sentado, e um
    palpite baixo demais custa cobertura silenciosamente enquanto um alto demais encerra
    a execução. A máquina vai dizer, então ela é perguntada.
    """
    try:
        with open("/proc/meminfo") as fh:
            info = {k.strip(): v for k, v in
                    (l.split(":", 1) for l in fh if ":" in l)}
        return int(info["MemAvailable"].split()[0]) / 1024 ** 2
    except Exception:
        return CACHE_BUDGET_GB / CACHE_FRACTION      # cai de volta pra constante antiga


def plan_cache(n_study, n_test=0):
    """Escolhe quantas slices por slot a memória que a máquina tem vai permitir.

    O cache é n_study x n_slot x slices x IMG^2 bytes. Cobertura é o eixo barato -
    linear - e resolução o caro, então quando o orçamento aperta é a contagem de slices
    que cede, não a grade de pixels. Decidir uma única vez, a partir do tamanho do corpus
    de treino, mantém os caches de treino e teste no mesmo layout de grupo.

    Só uma fração do que está livre é tomada. O resto não é folga: o encoder, suas
    ativações, os batches pinados e os frames vêm todos do mesmo pool, e o cache é a
    única alocação grande o bastante pra que estourá-la mate a execução de vez.
    """
    avail = available_gb()
    budget = min(avail * CACHE_FRACTION, CACHE_BUDGET_MAX_GB)
    # Os dois caches ficam retidos ao mesmo tempo, e a metade de teste é o que a execução
    # visível não consegue mostrar: aqui é um punhado de estudos, e na pontuação é o
    # conjunto oculto inteiro. Dimensionar só contra o corpus de treino, portanto, passa
    # em toda execução que pode ser observada e estoura a que conta.
    n_total = n_study + max(n_test, int(TEST_SHARE * n_study))
    per_slice = n_total * N_SLOT * IMG * IMG
    afford = int(budget * 1024 ** 3 // max(per_slice, 1))
    groups = max(1, min(N_GROUP_MAX, afford // GROUP))
    log(f"memory: {avail:.1f} GB available, {budget:.1f} GB to the cache; "
        f"sizing for {n_study} train + {n_total - n_study} test studies "
        f"-> {groups} group(s) of {GROUP} = {groups * GROUP} slices per slot"
        + (f" (wanted {N_GROUP_MAX})" if groups < N_GROUP_MAX else ""))
    return groups


N_GROUP = plan_cache(len(pd.read_csv(ROOT / "train.csv")),
                     len(pd.read_csv(ROOT / "test.csv")))
CACHE_SLICES = GROUP * N_GROUP
log(f"cache layout: {N_GROUP} groups x {GROUP} slices = {CACHE_SLICES} per slot")


In [ ]:
HDR_TAGS = ["SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",
            "RepetitionTime", "EchoTime", "Laterality", "PixelSpacing", "Rows",
            "Columns", "RescaleSlope", "RescaleIntercept",
            # Posição e orientação são lidas do mesmo header que probe() já abre, então
            # não custam nada, e são o que recupera o lado quando a tag Laterality está
            # ausente - o que acontece pra metade dos estudos aqui.
            "ImagePositionPatient", "ImageOrientationPatient"]


def _hdr_vec(s, n):
    """Faz parse de uma string DICOM multi-valor como armazenada por probe(): floats
    unidos por `|`."""
    if not isinstance(s, str):
        return None
    try:
        v = [float(x) for x in s.split("|")]
    except ValueError:
        return None
    return np.array(v) if len(v) >= n else None


def side_from_geometry(h):
    """Estudo -> 'L' / 'R' / None, a partir de onde a imagem se situa no paciente.

    `Laterality` (0020,0060) é Type 2C e legitimamente pode estar ausente; neste corpus
    ela falta em exatamente metade dos estudos, e os fabricantes de que ela falta são
    fabricantes inteiros, não séries espalhadas. Um estudo sem a tag não é um joelho
    esquerdo, mas a normalização a montante o trata como um, então metade do corpus nunca
    foi normalizada e os cinco alvos definidos por lado - os dois meniscos, os dois
    compartimentos tibiofemorais e o ligamento colateral medial - tiveram esse eixo
    invertido numa grande minoria dele.

    O sistema de coordenadas do paciente resolve isso sem a tag: +x é a esquerda do
    paciente, então o centro de um joelho direito fica em x negativo. O centro é usado em
    vez do próprio `ImagePositionPatient` porque esse é o canto da imagem, que é deslocado
    por metade de um campo de visão - o bastante pra mudar o sinal num joelho perto da
    linha média.

    É a mediana sobre as séries de um estudo que é limiarizada, não uma única série:
    probe() lê uma slice arbitrária por série, que numa pilha sagital pode estar em
    qualquer ponto da articulação. Estudos cujo centro cai perto da linha média ficam sem
    resolução em vez de serem adivinhados - medida contra a metade com tag, a regra acerta
    97% das vezes no geral e não é melhor que o acaso dentro de 20 mm.
    """
    cx = {}
    for r in h.itertuples(index=False):
        ipp = _hdr_vec(getattr(r, "ImagePositionPatient", None), 3)
        iop = _hdr_vec(getattr(r, "ImageOrientationPatient", None), 6)
        ps = _hdr_vec(getattr(r, "PixelSpacing", None), 2)
        rows, cols = getattr(r, "Rows", None), getattr(r, "Columns", None)
        if ipp is None or iop is None or ps is None or not rows or not cols:
            continue
        try:
            c = ipp[:3] + iop[:3] * ps[1] * float(cols) / 2 + iop[3:6] * ps[0] * float(rows) / 2
        except (TypeError, ValueError):
            continue
        cx.setdefault(r.StudyInstanceUID, []).append(float(c[0]))
    out = {}
    for st, xs in cx.items():
        m = float(np.median(xs))
        out[st] = None if abs(m) < LAT_MIN_OFFSET_MM else ("R" if m < 0 else "L")
    return out


def side_from_corner_x(h):
    """A lateralidade sob a qual um member importado foi ajustado.

    Limiariza a mediana do x bruto de `ImagePositionPatient` sobre as séries de um
    estudo. Esse é o x do *canto* da imagem, não do seu centro, então difere da regra
    acima por até metade de um campo de visão - o bastante pra reverter o sinal num
    joelho escaneado perto da linha média. A zona morta é 5 mm em vez de 20 mm, então
    também se compromete em estudos que a regra acima deixa sem resolução.

    Nenhuma das diferenças muda um shape. Cada uma decide se um estudo é espelhado, e um
    estudo espelhado de um jeito no treino e do outro na inferência apresenta os cinco
    alvos definidos por lado com o eixo invertido.
    """
    out = {}
    for st, g in h.groupby("StudyInstanceUID"):
        xs = []
        for r in g.itertuples(index=False):
            ipp = _hdr_vec(getattr(r, "ImagePositionPatient", None), 3)
            if ipp is not None and np.isfinite(ipp).all():
                xs.append(float(ipp[0]))
        if not xs:
            out[st] = None
            continue
        x = float(np.median(xs))
        # Coordenadas de paciente DICOM são LPS: +x é a esquerda do paciente.
        out[st] = None if abs(x) < LEGACY_LAT_OFFSET_MM else ("R" if x < 0 else "L")
    return out


def lat_of(h, tag=""):
    """Estudo -> 'L' / 'R' / None: a tag onde existe, geometria onde não existe.

    A tag está presente em exatamente metade dos estudos aqui e às vezes é uma string
    vazia em vez de ausente, o que não é o mesmo que NaN. Tratar a outra metade como do
    lado esquerdo é o que `normalise_laterality` fazia por omissão, então o fallback de
    geometria não é um refinamento - é a diferença entre normalizar metade do corpus e
    normalizar ele inteiro.
    """
    geo = side_from_corner_x(h) if RULES["lat"] == "corner_x" else side_from_geometry(h)
    d, n_tag, n_geo, n_none, n_disagree = {}, 0, 0, 0, 0
    for st, g in h.groupby("StudyInstanceUID"):
        v = [str(x).strip().upper() for x in g["Laterality"].dropna()]
        if RULES["lat"] == "corner_x" and "ImageLaterality" in g.columns:
            # A regra legacy também lê a segunda tag, então um estudo marcado só nela é
            # resolvido a partir da tag, não da geometria.
            v += [str(x).strip().upper() for x in g["ImageLaterality"].dropna()]
        v = [x[0] for x in v if x and x[0] in ("L", "R")]
        side = v[0] if v else None
        if side is not None:
            n_tag += 1
            if geo.get(st) is not None and geo[st] != side:
                n_disagree += 1
        else:
            side = geo.get(st)
            n_geo += side is not None
            n_none += side is None
        d[st] = side
    log(f"{tag}laterality: {n_tag} from the tag, {n_geo} from geometry, "
        f"{n_none} unresolved; tag and geometry disagree on {n_disagree} "
        f"({n_disagree / max(n_tag, 1):.1%} of the tagged)")
    return d



def probe(item):
    split, study, series, path = item
    row = {"split": split, "StudyInstanceUID": study, "SeriesInstanceUID": series,
           "dir": path}
    try:
        files = sorted(e.name for e in os.scandir(path) if e.name.endswith(".dcm"))
        row["files"] = files
        row["n_slices"] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]),
                             stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == "MultiValue":
                row[t] = "|".join(str(x) for x in v)
            else:
                row[t] = str(v)
    except Exception as exc:
        row["err"] = str(exc)[:120]
    return row


def walk(split):
    """Todo diretório de série de um split, com uma leitura de header por série.

    Um split ausente retorna um frame vazio *com as colunas que annotate espera*.
    Retornar um DataFrame nu parece a mesma coisa e não é: a próxima chamada indexa
    `SeriesDescription` e lança KeyError, então o branch que existe pra sobreviver a um
    split ausente é o que o transforma num crash.
    """
    base = ROOT / split
    items = []
    if not base.is_dir():
        return pd.DataFrame(columns=["split", "StudyInstanceUID", "SeriesInstanceUID",
                                     "dir", "files", "n_slices"] + HDR_TAGS)
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))
    return pd.DataFrame(rows)


def annotate(df):
    """Recupera supressão de gordura e ponderação da sequência de pulso a partir do header."""
    desc = (df["SeriesDescription"].fillna("") + " " + df["SequenceName"].fillna(""))
    desc = desc.str.lower().str.replace(_SEP, " ", regex=True)

    opts = df["ScanOptions"].fillna("").str.upper().str.split("|")
    # A GE escreve SAT_GEMS pra saturação espacial, então ScanOptions tem que ser casado
    # como tokens exatos; um teste de substring em "SAT" dispara em séries que não são
    # fat-sat.
    opts_fs = opts.apply(lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))
    df["fatsat"] = desc.str.contains(_FATSAT_RX) | opts_fs

    tr = pd.to_numeric(df["RepetitionTime"], errors="coerce")
    te = pd.to_numeric(df["EchoTime"], errors="coerce")
    gre = df["ScanningSequence"].fillna("").str.upper().str.contains("GR")
    t1, t2, pdw = desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX)

    df["weight"] = np.where(t1 & ~t2 & ~pdw, "T1",
                     np.where(t2 & ~pdw, "T2",
                       np.where(pdw, "PD",
                         np.where(gre, "GRE",
                           np.where(tr < 800, "T1",
                             np.where(te > 60, "T2",
                               np.where(tr >= 800, "PD", "UNK")))))))
    df["fluid"] = np.isin(df["weight"], ["PD", "T2"])
    df["px"] = pd.to_numeric(
        df["PixelSpacing"].fillna("").str.split("|").str[0].replace("", np.nan),
        errors="coerce")
    return df


In [ ]:
def pick_slots(series_df, plane_map):
    """Uma série por slot por estudo.

    Empates são resolvidos em favor da pilha com mais slices: uma pilha mais espessa
    amostra a articulação mais densamente, e o amostrador de três slices abaixo se
    beneficia da margem.
    """
    series_df = series_df.copy()
    series_df["plane"] = series_df["SeriesInstanceUID"].map(plane_map)
    out = {}
    for study, g in series_df.groupby("StudyInstanceUID"):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g["plane"] == plane) & (g["fatsat"] == fs)
            # fluid=None significa "não condicionar à ponderação" - o esquema público,
            # onde a única flag fornecida serve pros dois eixos de uma vez.
            if fluid is not None:
                sel &= (g["fluid"] == fluid)
            cand = g[sel]
            # Um slot sem nenhuma série casando seu predicado fica vazio, e nenhum
            # substituto é admitido de um predicado vizinho. Relaxar a ponderação pra
            # preencher um slot T1 puxaria do pool que `SAG_FLUID_NOFS` seleciona, já que
            # esse pool é o que sobra quando a ponderação é descartada: sobre o corpus de
            # treino isso colocaria uma série em dois slots pra 2383 de 4407 estudos e
            # deixaria 56% do slot T1 segurando PD ou T2. A máscara de presença então
            # afirmaria uma sequência que nunca foi adquirida, e o softmax por diagnóstico
            # do §6 dividiria sua atenção entre dois slots idênticos, dando a uma
            # aquisição cerca do dobro do peso que ela carrega num estudo que tem as duas.
            # A máscara existe pra dizer que um slot está ausente, que é o que um slot
            # ausente é.
            if len(cand) == 0 and RULES["slot_fallback"] and fluid is False:
                # O relaxamento que o parágrafo acima rejeita, reproduzido porque um
                # member importado foi ajustado com seus slots T1 preenchidos desse jeito:
                # mais da metade dos estudos de treino desse member tinham um slot T1
                # segurando uma série que não é T1. Deixar esses slots vazios o
                # apresentaria a uma máscara de presença que ele nunca viu.
                cand = g[(g["plane"] == plane) & (~g["fatsat"])]
            if len(cand):
                chosen[name] = cand.sort_values("n_slices", ascending=False).iloc[0]
        out[study] = chosen
    return out


## 3b. Em que ordem as slices estão

Uma série é um diretório de arquivos, e o jeito óbvio de percorrê-lo é ordenar pelos
nomes dos arquivos. Isso está errado aqui, e errado de um jeito que não produz erro
nenhum. O nome do arquivo é o SOP Instance UID, atribuído pra ser único em vez de
ordenado, então ordenar por ele produz uma sequência sem correlação com a anatomia —
medido numa série deste corpus, a correlação de rank entre a ordem por nome de arquivo e
a posição física é $\rho \approx 0.01$. Três coisas a jusante dependem dessa ordem: três
slices adjacentes como três canais viram três cortes transversais sem relação entre si;
o meio da pilha vira um subconjunto aleatório; e inverter a ordem pra normalizar
lateralidade não faz absolutamente nada.

A ordem verdadeira é recuperável exatamente a partir da geometria que toda slice
carrega. `ImageOrientationPatient` dá os eixos no plano $\hat{r}_x, \hat{r}_y$ e
`ImagePositionPatient` a posição $p$ do primeiro voxel, então

$$\hat{n} \;=\; \hat{r}_x \times \hat{r}_y, \qquad k \;=\; p \cdot \hat{n},$$

e $k$ aumenta monotonicamente ao longo da pilha. Como $k$ tem sinal e é expresso em
coordenadas de paciente, a pilha tem uma direção fixa ao longo do eixo esquerda-direita
do corpo — que é o que a normalização de lateralidade do §5 inverte. `InstanceNumber` é
o fallback onde as tags de geometria estão ausentes; costuma acompanhar $k$ a menos de
sinal, mas aquisições intercaladas e multi-eco numeram slices numa ordem que não é a
ordem que elas ocupam no espaço, e não tem sinal em coordenadas de paciente.


## 4. Amostragem: quantos milímetros um pixel pode valer

Uma slice DICOM de $N \times N$ pixels com espaçamento $s$ mm/pixel cobre $Ns$ milímetros
de anatomia. As duas variam por este corpus, então um resize de pixel fixo entrega ao
encoder imagens cuja escala física difere por um fator de várias vezes. Mas normalizar
escala é só metade da questão; a outra metade é um limite rígido.

**Um traço mais estreito que dois pixels não sobrevive ao resize.** Pra representar uma
estrutura de largura $d$ milímetros o passo de pixel precisa satisfazer $s_{\text{eff}}
\le d/2$, a condição de Nyquist aplicada à grade de reamostragem. Uma ruptura meniscal
tem de um a três milímetros, então em $d = 1$ mm o passo tem que ser no máximo $0.5$ mm —
e se não for, nenhuma capacidade a jusante recupera o sinal, porque ele já foi destruído
antes da primeira convolução. Isso é uma propriedade do resize, não da rede.

Cortar pra uma extensão física constante $L$ e reamostrar pra $P$ pixels fixa o passo:

$$n \;=\; \Big\lfloor \frac{L}{s} \Big\rceil \ \text{pixels}, \qquad
s_{\text{eff}} \;=\; \frac{L}{P}\ \ \text{mm/pixel}, \qquad
\text{token} \;=\; 14\,s_{\text{eff}}\ \ \text{mm}.$$

O corte tem que ser menor que o menor campo de visão, senão não faz nada
silenciosamente: se $L/s$ excede a largura da imagem o corte não pode ser feito e aquela
série passa sem normalização, sem nenhum erro. $L = 130$ mm fica abaixo do campo de
visão adquirido de quase toda série aqui e ainda contém a articulação. O alvo do resize
então decorre da largura da ruptura em vez de convenção — em $L = 130$ mm um input de
$224$ dá $0.580$ mm/pixel, acima do limite pra um traço de 1 mm, enquanto $336$ dá
$0.387$ mm/pixel e coloca um token de patch de $14$ pixels em $5.4$ mm.

**Intensidade precisa do mesmo tratamento**, já que ressonância não tem escala absoluta.
Cada série é normalizada pelos próprios percentis 1 e 99 — sobre a pilha amostrada em vez
de por slice, pra que as slices mantenham seu contraste relativo, e por percentis em vez
de extremos, pra que um vaso brilhante não comprima tudo o mais.


In [ ]:
ORDER_TAGS = [(0x0020, 0x0032), (0x0020, 0x0037), (0x0020, 0x0013)]

# Séries em que pelo menos uma slice amostrada não decodificaria. Uma lista em vez de um
# contador porque anexar é atômico sob as threads leitoras, e reportado em vez de
# engolido: sem reportar, uma falha de decode é indistinguível de um joelho preto.
DECODE_FAILED = []


def cache_tag(rules=None):
    """O nome sob o qual um cache decodificado é armazenado.

    Precisa nomear tudo que decide os pixels, não só suas dimensões. Duas configurações
    que concordam em resolução, contagem de slices, corte e banda mas discordam em como
    uma slice é escolhida produzem arrays diferentes de shape idêntico - então uma tag
    construída só a partir das dimensões deixa a segunda se anexar ao arquivo da primeira
    e treinar contra pixels que nunca pediu, sem nada em lugar nenhum reportando um
    descompasso.

    Uma leitura native mantém o nome simples, então caches decodificados antes das regras
    existirem continuam válidos; qualquer outra coisa ganha um sufixo.
    """
    r = dict(RULES if rules is None else rules)
    t = (f"{CACHE_IMG}px_{CACHE_SLICES}sl_{int(CROP_MM)}mm_"
         f"{SLICE_BAND[0]:.2f}-{SLICE_BAND[1]:.2f}")
    if {k: r.get(k, v) for k, v in RULES_NATIVE.items()} != RULES_NATIVE:
        t += "_" + hashlib.md5(json.dumps(r, sort_keys=True).encode()).hexdigest()[:6]
    return t


def _natural_key(name):
    return tuple(int(x) if x.isdigit() else x.lower()
                 for x in re.split(r"(\d+)", str(name)))


def _order_dominant_axis(rec):
    """A ordem de slice sob a qual um member importado foi ajustado.

    Ordena pela coordenada bruta de paciente ao longo de qualquer eixo que mais varia
    pela pilha, em vez da projeção sobre a normal da slice. As duas diferem por um sinal,
    não por uma fórmula: medido sobre este corpus, toda série sagital tem uma normal de
    slice com n_x em [-1.00, -0.98], então p.n é o negativo do x bruto pelo qual isso
    ordena e as duas pilhas saem exatamente invertidas. Como o amostrador de banda trunca
    em vez de arredondar, seus nove índices não são simétricos em torno do meio, então
    nove slices tiradas de uma pilha de vinte e seis compartilham duas entre uma ordem e
    outra.

    Geometria ausente recorre a `InstanceNumber` e depois a uma ordenação natural do nome
    do arquivo, ambas no mesmo limiar de 80% que o pipeline importado usava.
    """
    files, d = rec["files"], rec["dir"]
    rows = []
    for pos, f in enumerate(files):
        ipp = inst = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True,
                                 specific_tags=["ImagePositionPatient", "InstanceNumber"])
            raw = getattr(ds, "ImagePositionPatient", None)
            if raw is not None and len(raw) >= 3:
                c = np.asarray(raw[:3], dtype=np.float64)
                if np.isfinite(c).all():
                    ipp = c
            n = getattr(ds, "InstanceNumber", None)
            if n is not None:
                inst = float(n)
        except Exception:
            pass
        rows.append((f, ipp, inst, pos))

    placed = [r for r in rows if r[1] is not None]
    need = max(2, int(0.8 * len(rows)))
    if len(placed) >= need:
        xyz = np.stack([r[1] for r in placed])
        axis = int(np.argmax(np.ptp(xyz, axis=0)))
        spare = float(np.nanmedian(xyz[:, axis]))
        rows.sort(key=lambda r: (float(r[1][axis]) if r[1] is not None else spare,
                                 r[2] if r[2] is not None else float("inf"), r[3]))
    elif sum(r[2] is not None for r in rows) >= need:
        rows.sort(key=lambda r: (r[2] if r[2] is not None else float("inf"), r[3]))
    else:
        rows.sort(key=lambda r: _natural_key(r[0]))
    return [r[0] for r in rows], True


def order_slices(rec):
    """Retorna os arquivos da série ordenados ao longo do eixo through-plane.

    Um nome de arquivo DICOM aqui é um SOP Instance UID, atribuído arbitrariamente.
    Ordenar por ele portanto produz uma ordem sem correlação com anatomia - medido numa
    série, o Spearman entre o rank do nome de arquivo e a posição física é 0.009, ou
    seja, nenhuma. Qualquer coisa que assume que a ordem de arquivo significa algo está
    então operando sobre ruído: os três canais de um input "2.5D" são três visões sem
    relação entre si em vez de slices vizinhas, "o meio da pilha" é um subconjunto
    aleatório, e inverter a ordem das slices pra normalizar lateralidade não inverte nada
    que faça sentido.

    A ordem física é recuperável exatamente. Cada slice carrega sua posição em
    coordenadas de paciente e os eixos no plano; projetar a posição sobre a normal da
    slice dá uma coordenada through-plane com sinal, monotônica ao longo da pilha:

        n = r_x  x  r_y ,      k = p . n

    `InstanceNumber` é o fallback. Costuma acompanhar a projeção a menos de sinal, mas
    aquisições intercaladas e multi-eco não precisam numerar slices na ordem que ocupam
    no espaço - já a projeção tem sinal em coordenadas de paciente, que é o que a
    normalização de lateralidade precisa.
    """
    if RULES["order"] == "dominant_axis":
        return _order_dominant_axis(rec)
    files, d = rec["files"], rec["dir"]
    keyed = []
    for f in files:
        k = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True,
                                 specific_tags=ORDER_TAGS)
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            k = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            try:
                k = float(ds.InstanceNumber)
            except Exception:
                k = None
        keyed.append((k, f))
    if any(k is None for k, _ in keyed):
        # Uma série sem geometria utilizável mantém sua ordem arbitrária; isso é pior que
        # ordenar mas melhor que descartar a série, e é registrado em log como uma
        # contagem.
        return files, False
    return [f for _, f in sorted(keyed, key=lambda t: t[0])], True


def read_slot(rec, n_slice=None, out_size=None):
    """`n_slice` slices fisicamente espalhadas de uma série, em `out_size` pixels.

    Retorna uint8 [n_slice, out, out] normalizado por série pelos percentis 1-99.
    Percentis em vez de min/max porque a intensidade em RM não tem escala absoluta e um
    único vaso brilhante comprimiria toda a faixa dinâmica.

    Ler é a metade cara deste pipeline, então quem chama lê uma vez na maior configuração
    de que precisa e deriva as menores a partir do buffer retornado em vez de reler.
    """
    n_slice = GROUP if n_slice is None else n_slice
    out_size = IMG if out_size is None else out_size
    files, d, px = rec.get("ordered") or rec["files"], rec["dir"], rec["px"]
    n = len(files)
    if n == 0:
        return None
    # Espalha as amostras sobre uma banda central da pilha: as slices mais externas de
    # uma série de joelho são principalmente tecido mole fora da articulação. A banda é
    # uma constante em vez de um literal porque quanto da pilha vale a pena ler depende
    # de quantas slices estão sendo tomadas - com três, só o meio cabe, enquanto com
    # dezesseis as pontas valem a pena, e um cisto de Baker fica na ponta posteromedial
    # de uma pilha sagital.
    lo, hi = int(SLICE_BAND[0] * (n - 1)), int(SLICE_BAND[1] * (n - 1))
    idx = np.unique(np.linspace(lo, hi, n_slice).astype(int)) if hi > lo else np.array([n // 2])
    while len(idx) < n_slice:
        idx = np.append(idx, idx[-1])

    planes = []
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, "RescaleSlope", 1) or 1)
            ic = float(getattr(ds, "RescaleIntercept", 0) or 0)
            a = a * sl + ic
        except Exception:
            a = None                      # nenhum shape é conhecido aqui; ver abaixo
        planes.append(a)

    # Uma slice que não decodificaria não tem shape próprio, e inventar um é como um
    # único arquivo ilegível apaga uma série inteira: um substituto alocado no tamanho do
    # resize enquanto as slices decodificadas ainda estão no tamanho nativo faz a
    # checagem de shape abaixo tomar o substituto como autoridade e zerar as slices boas
    # junto, deixando um slot preto que a máscara de presença ainda reporta como
    # adquirido.
    #
    # Uma falha é em vez disso preenchida a partir da slice mais próxima que decodificou
    # - a mesma convenção que o amostrador já usa quando a banda tem menos slices
    # distintas do que foi pedido - e uma série onde nada decodifica é reportada como
    # ausente, o que a máscara consegue expressar, em vez de preta, o que ela não
    # consegue.
    got = [k for k, p in enumerate(planes) if p is not None]
    if RULES["decode_fill"] == "zero":
        # O que um member importado foi ajustado com: uma falha vira um plano zero no
        # tamanho do resize, que a checagem de shape abaixo então propaga pro slot
        # inteiro. É o comportamento que o parágrafo acima descreve e rejeita, mantido
        # aqui só porque os pesos daquele member foram aprendidos contra slots apagados
        # dessa maneira.
        if not got:
            DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        planes = [np.zeros((out_size, out_size), np.float32) if p is None else p
                  for p in planes]
        got = list(range(len(planes)))
    if not got:
        DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        return None
    if len(got) < len(planes):
        DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        for k, p in enumerate(planes):
            if p is None:
                planes[k] = planes[min(got, key=lambda j: abs(j - k))]

    # Slices de uma série ainda podem diferir em tamanho de matriz - multi-eco e alguns
    # reformats fazem isso - e essas genuinamente não são empilháveis.
    shp = planes[0].shape
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)

    # extensão física constante, depois resize: PixelSpacing varia 3.4x pelo corpus
    if px and np.isfinite(px) and px > 0:
        want = int(round(CROP_MM / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = h // 2, w // 2
            half = want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]

    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)

    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode="bilinear", align_corners=False)
    # uint8, não float32. Esses buffers se enfileiram entre as threads leitoras e o
    # encoder, e neste tamanho uma slot-série em float32 tem vários megabytes. A
    # intensidade já está normalizada em [0, 1] aqui, então oito bits não custam nada que
    # um resize bilinear já não tenha custado, e a fila fica com um quarto do tamanho.
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)


## 5. Normalizando esquerda e direita

Quatro dos doze alvos — os dois meniscos e os compartimentos tibiofemorais medial e
lateral — são pares medial/lateral, e um quinto, o ligamento colateral medial, leva o
nome do lado em que fica. Medial e lateral são definidos em relação à linha média do
corpo, então de que lado da *imagem* eles caem depende de qual joelho foi escaneado. A
menos que isso seja normalizado, esses cinco labels aprendem a partir de um eixo que o
modelo não consegue observar.

A correção difere por plano. Coronal e axialmente a direção medial-lateral está no
plano da imagem, então inverter o último eixo mapeia um joelho no outro.
Sagitalmente é o eixo da *slice*: cada slice fica inalterada por espelhamento, e o que
difere é a ordem em que a pilha percorre a articulação, então a ordem das slices é
invertida em vez disso.

`Laterality` é um atributo Type 2C: pode legitimamente estar ausente, e aqui está ausente
em metade dos estudos — por fabricantes inteiros, não séries espalhadas. Deixar esses de
lado declara silenciosamente que são do lado esquerdo, então todo joelho direito entre
eles entra no modelo espelhado. O sistema de coordenadas do paciente fornece a tag que
falta: posição e orientação são registradas por imagem e $+x$ aponta pra esquerda do
paciente, então o sinal do $x$ do centro da imagem diz qual joelho é este.

$$c \;=\; \mathbf{p} \;+\; \mathbf{r}\,\Delta_c \frac{N_c}{2} \;+\; \mathbf{d}\,\Delta_r \frac{N_r}{2},
\qquad \text{lado} = \begin{cases} \text{direito} & c_x < 0\\ \text{esquerdo} & c_x > 0\end{cases}$$

com $\mathbf{p}$ a posição da imagem, $\mathbf{r}$ e $\mathbf{d}$ os cossenos diretores de
linha e coluna, e $\Delta$ o espaçamento de pixel. O centro é usado em vez de
$\mathbf{p}$ diretamente, que é um canto a meio campo de visão de distância — o
suficiente pra mudar o sinal num joelho escaneado perto da linha média. É a mediana sobre
as séries de um estudo que é limiarizada, e um estudo centrado a uma distância curta da
linha média fica sem resolução em vez de ser adivinhado, porque dentro dessa banda o
sinal não é melhor que o acaso.


In [ ]:
def normalise_laterality(img, plane, lat):
    """Mapeia todo joelho pra uma convenção de joelho esquerdo.

    Vistas coronais e axiais espelham sob um flip horizontal. Pilhas sagitais não são
    imagens espelhadas uma da outra - a ordem das slices corre medial-lateral em
    direções opostas - então a ordem dos canais é invertida em vez disso.
    """
    if lat != "R":
        return img
    if plane in ("Coronal", "Axial"):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])


## 5b. Ler uma vez, treinar muitas vezes

O custo deste pipeline é dominado pela leitura, não pela aritmética. Um estudo tem
várias séries e cada série dezenas de slices, então um estudo é da ordem de umas cento e
cinquenta arquivos. Isso é pagável uma vez; não é pagável a cada época, e o fine-tuning
precisa dos mesmos pixels toda época. Então as imagens de slot são decodificadas uma
única vez pra memória e mantidas como `uint8`:

$$\text{bytes} \;=\; N_{\text{study}} \times N_{\text{slot}} \times S \times P^{2}$$

com $S$ slices mantidas por slot em $P$ pixels. O expoente em $P$ é o que torna isso uma
restrição real em vez de um detalhe — o cache cresce com o *quadrado* da resolução e só
linearmente com as slices, então cobertura é o eixo barato e resolução o caro.

As slices em cache formam um input de encoder de três canais por slot, e o layout
generaliza pra vários grupos assim por slot: o treino tira um por passo, o que funciona
como augmentation ao longo da pilha, e a inferência faz a média sobre eles. O que fixa o
número de grupos é um orçamento, não uma capacidade — o cache tem permissão pra usar uma
fração da memória reportada como livre, deliberadamente abaixo do total, já que é a
única alocação grande o bastante pra que estourá-la encerre a execução em vez de
desacelerá-la, e o encoder, suas ativações e os buffers em trânsito vêm do mesmo pool. O
tamanho comparado contra esse orçamento é a soma dos *dois* caches, treino e teste,
porque os dois ficam residentes ao mesmo tempo.

Um arquivo de submissão válido é escrito antes de qualquer coisa disso começar e
sobrescrito assim que predições reais existirem, então a execução sempre deixa um
arquivo pontuável pra trás.


In [ ]:
# Onde a ordem geométrica de slices pode ser lembrada entre execuções. Desligado na
# plataforma, porque cada execução ganha uma máquina nova e não há nada a lembrar; ligado
# fora dela, onde o mesmo corpus é colocado em cache de novo em cada resolução e
# contagem de slices e a ordem não é função de nenhuma das duas. É opt-in pra que o
# comportamento da execução pontuada seja decidido pelo código, não por um arquivo que
# por acaso esteja por aí.
ORDER_CACHE = os.environ.get("RSNA_ORDER_CACHE") or None


def build_cache(slot_map, plane_map, lat_map, tag):
    """Decodifica cada (study, slot) uma vez pra um array uint8 em memória.

    O fine-tuning revisita os mesmos pixels a cada época. Lê-los do mount toda vez
    tornaria a contagem de épocas função de I/O em vez de aprendizado, então são
    decodificados uma vez e mantidos como bytes: a intensidade já foi normalizada em
    [0, 1], e oito bits não custam nada que um resize bilinear já não tenha custado.

    CACHE_SLICES posições são mantidas por slot, que o loop de treino lê como N_GROUP
    grupos de GROUP canais consecutivos.
    """
    studies = sorted(slot_map)
    sidx = {s: i for i, s in enumerate(studies)}
    cache = np.zeros((len(studies), N_SLOT, CACHE_SLICES, IMG, IMG), np.uint8)
    mask = np.zeros((len(studies), N_SLOT), np.float32)
    log(f"{tag}: cache {cache.shape} = {cache.nbytes / 1024 ** 3:.1f} GB")

    jobs = [(st, k, plane, slot_map[st][name])
            for st in studies
            for k, (name, plane, _, _) in enumerate(SLOTS)
            if name in slot_map[st]]
    n_job = len(jobs)

    # Ordenação primeiro, e como passada própria. Lê um header por slice de cada série
    # escolhida - bem mais aberturas de arquivo que o decode que vem depois - e num
    # mount de rede isso é latência, não trabalho, então ganha seu próprio pool mais
    # largo.
    t_ord = time.time()
    n_slice_total = sum(len(j[3]["files"]) for j in jobs)
    log(f"{tag}: ordering {len(jobs)} slot-series ({n_slice_total} slice headers)")
    ok = done = 0
    CHUNK_O = 1024

    # Uma ordem lembrada, quando uma é oferecida. A projeção depende só da geometria
    # DICOM, então é a mesma em toda resolução e toda contagem de slices, e custa uma
    # leitura de header por slice - o maior custo isolado nesta passada. Uma entrada é
    # validada pelo número de arquivos presentes, então uma árvore que mudou por baixo
    # dela é recalculada em vez de confiada: ordem é dado derivado, e uma entrada
    # obsoleta seria invisível do jeito que mais importa.
    seen = {}
    if ORDER_CACHE and Path(ORDER_CACHE).is_file():
        try:
            import json as _json
            seen = _json.loads(Path(ORDER_CACHE).read_text())
        except (OSError, ValueError):
            seen = {}
        hit = 0
        for _, _, _, rec in jobs:
            e = seen.get(rec["SeriesInstanceUID"])
            if e and len(e["files"]) == len(rec["files"]):
                rec["ordered"] = e["files"]
                ok += int(e["good"])
                hit += 1
        jobs = [j for j in jobs if "ordered" not in j[3]]
        log(f"{tag}: {hit} slot-series ordered from {ORDER_CACHE}, {len(jobs)} to read")

    with ThreadPoolExecutor(max_workers=ORDER_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK_O):
            block = jobs[c0:c0 + CHUNK_O]
            for (_, _, _, rec), (files, good) in zip(
                    block, pool.map(lambda j: order_slices(j[3]), block)):
                rec["ordered"] = files
                ok += int(good)
                done += 1
                if ORDER_CACHE:
                    seen[rec["SeriesInstanceUID"]] = {"files": files, "good": bool(good)}
            # O teto é o que vier primeiro: o orçamento próprio da passada, ou a fração
            # do que resta da execução que ela pode tomar. O segundo é o que torna
            # seguro definir o primeiro generosamente - um mount lento o bastante pra
            # importar não consegue gastar o tempo de treino, porque o orçamento encolhe
            # junto com a execução.
            budget = min(ORDER_BUDGET_S, max(60.0, (TIME_BUDGET - (time.time() - T0)) * 0.35))
            if time.time() - t_ord > budget:
                log(f"{tag}: ordering budget spent at {done}/{len(jobs)}; "
                    f"the rest keep file order")
                break
    if ORDER_CACHE and done:
        import json as _json
        _t = Path(ORDER_CACHE).with_suffix(".tmp")
        _t.write_text(_json.dumps(seen))
        _t.replace(Path(ORDER_CACHE))
    log(f"{tag}: ordered {ok}/{n_job} by geometry "
        f"({n_job - ok} kept arbitrary) in {time.time() - t_ord:.0f}s")

    jobs = [(st, k, plane, slot_map[st][name])
            for st in studies
            for k, (name, plane, _, _) in enumerate(SLOTS)
            if name in slot_map[st]]
    log(f"{tag}: decoding {len(jobs)} slot-series")
    n_failed_before = len(DECODE_FAILED)

    CHUNK = 512
    done = 0
    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK):
            block = jobs[c0:c0 + CHUNK]
            for (st, k, plane, _), img in zip(
                    block, pool.map(lambda j: read_slot(j[3], CACHE_SLICES, IMG), block)):
                done += 1
                if img is None:
                    continue
                cache[sidx[st], k] = normalise_laterality(img, plane,
                                                          lat_map.get(st)).numpy()
                mask[sidx[st], k] = 1.0
            if done % 4096 < CHUNK:
                log(f"  {tag} {done}/{len(jobs)}")
            if time.time() - T0 > TIME_BUDGET:
                log(f"  {tag}: time budget reached during decode")
                break
    n_failed = len(DECODE_FAILED) - n_failed_before
    log(f"{tag}: {int(mask.sum())}/{len(jobs)} slots filled"
        + (f"; {n_failed} series had a slice that would not decode" if n_failed else ""))
    gc.collect()
    return studies, cache, mask


## 6. Agregando slots em doze decisões

Um estudo chega como até seis embeddings de slot $x_s \in \mathbb{R}^{d}$ com uma máscara
de presença $m_s \in \{0,1\}$. Fazer o pooling deles de forma idêntica descartaria o
motivo pelo qual o protocolo tem três planos: cada achado é lido em sequências
específicas, e uma média sobre slots dilui o que carrega a evidência com cinco que não
carregam.

Projeta cada slot, soma uma identidade de slot aprendida, dá a cada diagnóstico $o$ sua
própria query $q_o \in \mathbb{R}^{H}$, e deixa ele atender sobre os slots com os
ausentes mascarados fora do softmax:

$$h_s \;=\; \phi(x_s) + e_s, \qquad
\alpha_{o,s} \;=\; \frac{\exp\!\big(\langle h_s, q_o\rangle / \sqrt{H}\big)\, m_s}
{\sum_{s'} \exp\!\big(\langle h_{s'}, q_o\rangle / \sqrt{H}\big)\, m_{s'}},$$

$$c_o \;=\; \sum_s \alpha_{o,s}\, h_s, \qquad
\ell_o \;=\; \langle c_o, w_o \rangle + b_o .$$

O softmax mascarado renormaliza sobre o que o estudo de fato contém, então uma série
axial faltante desloca a atenção de um diagnóstico pras sequências presentes em vez de
alimentá-lo com um vetor zero.

**A cabeça (head) é deliberadamente pequena assim.** O label é atrelado ao *estudo*,
então nada na supervisão diz qual parte de um estudo carrega o achado, e uma agregação
mais rica não teria sinal nenhum de onde aprender isso. Onde a supervisão é grosseira, a
cabeça também é.


In [ ]:
class SlotHead(nn.Module):
    """Atenção por diagnóstico sobre os embeddings de slot de um estudo.

    Cada achado é lido em sequências específicas - cruzados sagitalmente, ligamentos
    colaterais e o corpo do menisco coronalmente, cartilagem patelar axialmente - então
    fazer o pooling dos slots de forma idêntica diluiria o que carrega a evidência com o
    resto.

    A agregação é deliberadamente simples assim. Com um label em nível de estudo, não há
    sinal dizendo ao modelo qual parte de um estudo importa, então parâmetros de atenção
    extras abaixo do nível de slot não teriam nada de onde aprender e gastariam sua
    capacidade ajustando ruído.
    """

    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2, prior=False):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        # Um member importado carrega uma inclinação fixa por (diagnóstico, slot) nos
        # logits de atenção, definida pela tabela de anatomia abaixo em vez de aprendida.
        # É um buffer, então viaja no state dict e precisa existir pra esse member
        # carregar; exp(0.55) dá a um slot preferido cerca de 1.73x o peso de um não
        # preferido, o que enviesa o softmax sem nunca excluir um slot.
        p_ = torch.zeros(n_out, n_slot)
        if prior and n_slot == len(SLOTS) and n_out == len(TARGETS):
            for t, slots in SLOT_PRIOR_TABLE.items():
                if t in TARGETS:
                    p_[TARGETS.index(t), list(slots)] = SLOT_PRIOR_STRENGTH
        self.prior = prior
        if prior:
            self.register_buffer("slot_prior", p_)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        att = torch.einsum("bsh,oh->bos", h, self.query) / self.hidden ** 0.5
        if self.prior:
            att = att + self.slot_prior.unsqueeze(0)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        ctx = self.drop(torch.einsum("bos,bsh->boh", att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


In [ ]:
class Model(nn.Module):
    """Encoder mais head, treinados de ponta a ponta.

    Um estudo chega como uma bag de imagens de slot. A bag é achatada pro encoder e
    dobrada de volta antes da head, então o encoder nunca vê a estrutura do estudo e a
    head nunca vê pixels.
    """

    def __init__(self, backbone, dim, pool="cls_mean", prior=False):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.head = SlotHead(dim * POOL_PARTS[pool], N_SLOT, len(TARGETS), prior=prior)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask, img_size=None):
        B, S = imgs.shape[:2]
        x = imgs.reshape(B * S, *imgs.shape[2:]).float().div_(255.0)
        if img_size is not None and img_size != x.shape[-1]:
            # O cache é mantido na maior resolução que qualquer configuração precisa; o
            # resto reduz a partir dele, então toda configuração vê os mesmos pixels
            # através de uma grade de amostragem diferente em vez de um corte diferente.
            x = F.interpolate(x, size=(img_size, img_size), mode="bilinear",
                              align_corners=False)
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state
        patch = out[:, 1:]
        parts = [out[:, 0], patch.mean(1)]
        if self.pool == "cls_mean_focal":
            # A cauda superior de cada canal sobre a grade de patches, tomada por canal
            # em vez de selecionando patches inteiros: um achado ocupa uma pequena parte
            # do campo, então uma média simples sobre 256 patches o dilui em duas ordens
            # de grandeza, e isso mantém o oitavo superior das respostas de cada canal.
            k = max(1, patch.shape[1] // 8)
            parts.append(patch.topk(k, dim=1).values.mean(1))
        feat = torch.cat(parts, dim=1).reshape(B, S, -1)
        return self.head(feat, mask)


### Por que o encoder é treinado em vez de congelado

Um encoder auto-supervisionado congelado é limitado por algo que nenhum trabalho a
jusante consegue alcançar. Resolução, tamanho do encoder, cobertura de slices e
agregação de slots mudam o quanto o modelo olha e com que atenção, mas nenhum muda o
vocabulário com que ele olha, então todos esses eixos esbarram no mesmo teto — e esse
teto deveria mesmo se esperar que vinculasse aqui, já que o encoder aprendeu suas
features a partir de imagens naturais, onde nada se parece com o sinal que uma ruptura
meniscal faz numa sequência ponderada em densidade de prótons.

Então o encoder é adaptado, com duas restrições. **Só os últimos blocos se movem** — os
blocos iniciais de um vision transformer são filtros genéricos de borda e textura, os
blocos finais são onde a semântica mora, e pode não haver supervisão suficiente aqui pra
melhorar os iniciais enquanto certamente há o bastante pra danificá-los. **O encoder
aprende bem mais devagar que a head** — a head está aleatória na inicialização e tem
tudo pra aprender, o encoder parte de uma boa solução e só precisa ser deslocado dela,
então uma única taxa de aprendizado deixaria a head sem treino ou destruiria o encoder
nos primeiros centenas de passos.

Os alvos continuam sendo os labels derivados de laudo do §2, ponderados pela confiança
por achado que vem junto, então um laudo que nunca menciona um achado puxa fraco naquela
saída em vez de afirmar um negativo ali. Os estudos com anotações por condição são
ponderados acima de toda linha derivada, já que são os únicos labels lidos das imagens.


In [ ]:
def build_model(unfreeze_last, source=None, variant="small", pool="cls_mean",
                prior=False):
    """Carrega o encoder e abre os últimos `unfreeze_last` blocos pro treino.

    Os blocos iniciais de um transformer auto-supervisionado são filtros genéricos de
    borda e textura; os blocos finais carregam semântica. Abrir só os finais é a escolha
    cautelosa - pode não haver supervisão suficiente aqui pra melhorar os iniciais e
    certamente há o bastante pra danificá-los - mas até onde a linha deveria ficar é uma
    pergunta que o corpus tem que responder, não a intuição.

    `source` nomeia de onde vêm os pesos. Deixado sem definir, é o diretório de modelo
    anexado, que é a única coisa disponível aqui. É um parâmetro pra que uma execução
    fora da plataforma construa o mesmo objeto a partir do mesmo código em vez de uma
    segunda definição que tem que ser mantida sincronizada à mão.
    """
    from transformers import AutoModel
    p = source if source is not None else find_dinov2(variant)
    if p is None:
        raise FileNotFoundError("DINOv2 weights not attached")
    bb = AutoModel.from_pretrained(str(p))
    n_layer = len(bb.encoder.layer)
    for prm in bb.parameters():
        prm.requires_grad = False
    for blk in bb.encoder.layer[max(0, n_layer - unfreeze_last):]:
        for prm in blk.parameters():
            prm.requires_grad = True
    for prm in bb.layernorm.parameters():
        prm.requires_grad = True
    dim = bb.config.hidden_size
    trainable = sum(p.numel() for p in bb.parameters() if p.requires_grad)
    log(f"backbone: {n_layer} blocks, last {unfreeze_last} trainable "
        f"({trainable / 1e6:.1f}M params), feature dim {dim * POOL_PARTS[pool]}")
    return Model(bb, dim, pool=pool, prior=prior)


## 6b. Lendo pesos em vez de aprendê-los

Nada exige que a execução pontuada seja a execução que aprendeu os pesos. Um notebook
pode anexar um dataset, pesos são um dataset, e a parte que genuinamente não pode ser
feita antecipadamente é a parte que depende de estudos que ninguém viu — lê-los, e
prever. Aprender aqui em vez disso limita o modelo ao que um único acelerador consegue
caber dentro do limite de tempo, e gasta esse tempo de novo a cada submissão pra um
resultado que não muda. Então, quando um pacote de members treinados está anexado, esta
seção o lê; quando nenhum está, as seções abaixo treinam um.

**Members, não um modelo.** Cada member carrega o pré-processamento com o qual foi
ajustado. Dois members ajustados em resoluções diferentes não podem compartilhar um
decode; dois ajustados de forma parecida podem. Então members são agrupados pelos
pixels de que precisam, cada grupo é decodificado uma vez, e um member adicionado depois
se junta à lista sem alteração.

O que os agrupa é mais do que a resolução e o corte. Quatro outras decisões definem o
que uma slice *é* — qual vem a seguir ao longo da pilha, quais joelhos são espelhados,
se um slot pode ser preenchido a partir de uma sequência que não casa com seu predicado,
e o que substitui uma slice que não vai decodificar — e nenhuma delas muda um único
shape. Um member ajustado sob uma leitura e decodificado sob outra, portanto, carrega
sem problema, roda, e retorna uma submissão calculada a partir da imagem errada, então a
leitura viaja com o member.

**Um member tem que provar que é o modelo que diz ser.** Carregar um state dictionary
tem sucesso sempre que os shapes batem, e shapes batem em toda diferença que importa -
uma normalização, resize, ou banda de slices alterados. Nenhuma dessas levanta erro.
Então cada member carrega a resposta que deu a uma pergunta com seed, recalculada antes
do uso; um descompasso para a execução em vez de ser incluído numa média na submissão.

**Como um member é lido.** O treino tira um grupo de slices consecutivas por passo. Na
inferência, olhar mais não custa decode extra nenhum — o cache já mantém $S$ slices,
então um grupo do meio é $1$ passada forward, os grupos disjuntos são $S/G$, e toda
sequência consecutiva de $G$ slices é $S-G+1$. Onde a média é tomada é de graça mas não é
neutro: fazer a média dos logits e depois espremer é uma média geométrica de odds, fazer
a média das probabilidades é uma média aritmética de risco, e elas ordenam os estudos de
forma diferente. Members são combinados por rank, já que o §1 estabeleceu que ordem é
tudo que a métrica lê.


In [ ]:
FINGERPRINT_TOL = 2e-3


def fingerprint(model, dev, img_size, n_slot=None, group=None, seed=None):
    """A saída do modelo numa bag sintética fixa, como uma identidade portável.

    Pesos que são carregados mas lidos através do pré-processamento errado produzem
    predições, não erros. A submissão é bem formada, o log não diz nada, e a diferença é
    um número que nenhuma saída da execução revela. Um scaling que nunca acontece, ou
    acontece duas vezes, já basta por si só e não muda shape nenhum em lugar nenhum.

    Então um conjunto de pesos carrega a resposta que deu a uma pergunta sem nenhum dado
    nela. O input é gerado a partir de uma seed em vez de lido, então é o mesmo em
    qualquer máquina, e é empurrado pelo caminho forward inteiro - o scaling de bytes, a
    normalização ImageNet, o resize, o encoder, a atenção de slot. Qualquer uma dessas
    coisas diferindo move a saída por ordem um. Diferenças numéricas entre duas GPUs a
    movem por cerca de 1e-5, que é por isso que a tolerância fica entre os dois em vez de
    em zero.

    Isso checa que o modelo calcula o que calculava quando foi ajustado. Não consegue
    checar que os pixels que chegam até ele são os pixels certos; `read_slot` e a
    passada de header respondem por seus próprios testes.
    """
    n_slot = N_SLOT if n_slot is None else n_slot
    group = GROUP if group is None else group
    seed = SEED if seed is None else seed
    g = torch.Generator().manual_seed(seed)
    imgs = torch.randint(0, 256, (2, n_slot, group, img_size, img_size),
                         generator=g, dtype=torch.uint8).to(dev)
    mask = torch.ones(2, n_slot, device=dev)
    mask[1, -1] = 0.0                       # exercita o branch mascarado do softmax
    was_training = model.training
    model.eval()
    with torch.no_grad():
        # float32 do início ao fim: autocast faria o valor depender de qual dispositivo
        # calhou de rodá-lo, e o ponto do número é que ele não depende.
        out = model(imgs, mask, img_size).float().cpu().numpy()
    if was_training:
        model.train()
    return out


def check_fingerprint(model, dev, img_size, expected, tol=FINGERPRINT_TOL, tag=""):
    """Compara contra um fingerprint armazenado; lança erro quando o modelo não é o
    mesmo mapa."""
    got = fingerprint(model, dev, img_size)
    exp = np.asarray(expected, np.float32)
    if got.shape != exp.shape:
        raise WeightsError(f"{tag}fingerprint shape {got.shape} != stored {exp.shape}: "
                           f"the architecture is not the one these weights were fitted to")
    d = float(np.abs(got - exp).max())
    if d > tol:
        raise WeightsError(
            f"{tag}fingerprint differs by {d:.4g} (tolerance {tol:g}). The weights load "
            f"but do not compute what they computed when fitted - preprocessing, "
            f"resolution or architecture has moved between the two runs.")
    log(f"{tag}fingerprint matches within {d:.2g}")
    return d


class WeightsError(RuntimeError):
    """Lançada quando pesos anexados não podem ser confiados como os que foram ajustados.

    Deliberadamente fatal pelo mesmo motivo que LabelSourceError: uma execução que prevê
    a partir de um modelo descasado completa, escreve uma submissão plausível, e difere
    de uma correta só num número que nenhuma saída da execução revela.
    """


def find_weights(name="manifest.json"):
    """Localiza um pacote de pesos montado, ou retorna None se nenhum estiver anexado.

    Mesmo formato de `find_label_table`: o notebook precisa continuar funcionando pra um
    leitor que não anexa nada, então ausência é um caminho, não um erro. O que não pode
    ficar silencioso é um pacote que está anexado e inutilizável, e é isso que
    `load_weights` recusa.
    """
    import json
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
        if name not in files:
            continue
        # O manifesto decide, não os nomes de arquivo ao lado dele. Testar por uma
        # convenção de nomenclatura faz a busca concordar com o que quer que quem
        # empacotou tenha chamado seus arquivos por último, o que é uma segunda
        # definição do que um pacote é.
        try:
            man = json.loads((Path(root) / name).read_text())
        except (OSError, ValueError):
            continue
        if isinstance(man.get("members"), list) and man["members"]:
            missing = [m["file"] for m in man["members"]
                       if not (Path(root) / m["file"]).is_file()]
            if missing:
                raise WeightsError(
                    f"{root} holds a manifest listing {len(man['members'])} members but "
                    f"{len(missing)} of their files are absent (first {missing[0]!r})")
            return Path(root)
    return None


# Como um member é lido na inferência. Janelas sobrepostas sobre as slices que o cache
# já mantém custam passadas forward e nenhum decode extra, que é a direção barata pra
# gastar; e fazer a média das probabilidades em vez dos logits é uma média aritmética de
# risco em vez de uma média geométrica de odds, o que ordena os estudos de forma
# diferente. Os dois foram escolhidos medindo-os nos folds que cada member deixou de
# fora, não por argumento.
TTA_OVERLAP = True
TTA_POOL = "prob"


def window_starts(n_slice, group, overlap=None):
    """Onde cada janela de TTA começa."""
    overlap = TTA_OVERLAP if overlap is None else overlap
    if overlap and n_slice >= group:
        return list(range(n_slice - group + 1))
    return [g * group for g in range(max(n_slice // group, 1))]


@torch.no_grad()
def predict_member(model, cache, mask, idx, dev, img_size, group=None, pool=None,
                   starts=None):
    """As predições de um member, com a média sobre suas janelas de TTA.

    `starts` é um parâmetro em vez de sempre derivado aqui pra que quem chama, tendo
    medido o que o tempo restante permite, possa ler um member sobre menos janelas.
    Passá-lo é como a redução se torna visível num único lugar em vez de ser
    rederivada.
    """
    group = GROUP if group is None else group
    pool = TTA_POOL if pool is None else pool
    starts = window_starts(cache.shape[2], group) if starts is None else list(starts)
    if not starts:
        raise ValueError("predict_member was given no windows to average over")
    model.eval()
    out = []
    for b in range(0, len(idx), EVAL_BATCH):
        sel = idx[b:b + EVAL_BATCH]
        m = torch.from_numpy(mask[sel]).to(dev)
        acc = None
        for st in starts:
            rows = torch.from_numpy(
                np.ascontiguousarray(cache[sel, :, st:st + group])).to(dev)
            with torch.autocast("cuda", enabled=dev.type == "cuda"):
                z = model(rows, m, img_size).float()
            v = z if pool == "logit" else torch.sigmoid(z)
            acc = v if acc is None else acc + v
        v = acc / len(starts)
        out.append((torch.sigmoid(v) if pool == "logit" else v).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, len(TARGETS)), np.float32)


def infer_from_package(path, dev):
    """Prevê o split de teste a partir de um pacote anexado de members treinados.

    A alternativa abaixo - aprender os pesos dentro da execução pontuada - gasta o
    orçamento inteiro no corpus de treino toda vez que o notebook é submetido, e limita o
    modelo ao que nove horas num único acelerador conseguem caber. Nenhuma das duas é
    necessária: um notebook pode anexar um dataset, e pesos são um dataset. O que a
    execução pontuada então faz é a parte que não pode ser feita antecipadamente, porque
    os estudos não são conhecidos com antecedência.

    Members são agrupados pelos pixels de que precisam. Dois members ajustados em
    resoluções diferentes são funções diferentes do mesmo estudo e não podem
    compartilhar um decode; dois ajustados de forma parecida podem, e esse é o motivo
    inteiro do agrupamento existir em vez de um decode por member.
    """
    import json
    man = json.loads((Path(path) / "manifest.json").read_text())
    members = man["members"]
    log(f"weights package: {len(members)} member(s) from {path}")

    test_df = pd.read_csv(ROOT / "test.csv")
    test_series = pd.read_csv(ROOT / "test_series.csv")
    plane_map = dict(zip(test_series["SeriesInstanceUID"],
                         test_series["Anatomical_Plane"]))
    hte = annotate(walk("test_series"))
    log(f"test header pass: {len(hte)} series")

    groups = {}
    for m in members:
        groups.setdefault(m["pixel_group"], []).append(m)

    per_member = []
    fixed_s = per_win_s = None
    for gi, (key, gm) in enumerate(groups.items(), 1):
        cfg = json.loads(key)
        adopt_config_globals(cfg)
        log(f"decode group {gi}/{len(groups)}: {cfg['img']}px x {cfg['slices']} slices, "
            f"crop {cfg['crop_mm']} mm -> {len(gm)} member(s)")
        st_te, Cte, Mte = build_cache(pick_slots(hte, plane_map), plane_map,
                                      lat_of(hte, "test "), f"test g{gi}")
        idx = np.arange(len(st_te))

        # O que o tempo restante permite. Um member custa uma parte fixa - carregar o
        # checkpoint, construir o encoder, checar o fingerprint - mais uma parte que
        # escala com o número de janelas de TTA sobre as quais é lido. Só a segunda vale
        # a pena trocar, e só o primeiro member consegue medir qualquer uma delas, porque
        # o tamanho do conjunto de teste não é algo que o notebook recebe.
        #
        # As duas partes são cronometradas separadamente. Dividir o tempo de parede
        # inteiro de um member pela contagem de janelas dele precifica a parte fixa como
        # se ela escalasse, então toda redução infla a estimativa que a causou e a
        # estimativa foge da verdade.
        #
        # Janelas são abandonadas antes de members. Uma janela descartada custa uma visão
        # de um estudo que outras janelas também veem; um member descartado custa um voto
        # independente, que é do que um ensemble é feito. A pergunta feita antes de cada
        # member é se MAIS UM cabe - não se todo o resto cabe, o que abandonaria um
        # ensemble que ainda poderia ter rodado a maior parte de si mesmo.
        starts = window_starts(Cte.shape[2], GROUP)
        order = sorted(gm, key=lambda m: -(m.get("holdout") or 0))
        left_after = sum(len(g) for j, (_, g) in enumerate(groups.items(), 1) if j > gi)
        for k, m in enumerate(order):
            left = TIME_BUDGET - (time.time() - T0)
            remaining = (len(order) - k) + left_after
            if fixed_s is not None and per_win_s is not None:
                # Deixa um décimo do que resta sem gastar: a estimativa vem de um member
                # numa máquina, e uma submissão escrita tarde demais não é escrita.
                afford = max(left * 0.9, 0.0)
                need = fixed_s + len(starts) * per_win_s
                if need * remaining > afford:
                    room = afford / max(remaining, 1)
                    n_win = int((room - fixed_s) / per_win_s) if per_win_s > 0 else 0
                    n_win = max(1, min(len(starts), n_win))
                    if fixed_s + per_win_s > afford:
                        log(f"  {left / 60:.0f} min left: stopping after {k} of "
                            f"{len(order)} in this decode group; not one more member "
                            f"fits, at any window count")
                        break
                    if n_win < len(starts):
                        mid = (len(starts) - n_win) // 2
                        log(f"  {left / 60:.0f} min left, {remaining} member(s) to go: "
                            f"{n_win} window(s) each instead of {len(starts)}")
                        starts = starts[mid:mid + n_win]
            t0 = time.time()
            ck = torch.load(Path(path) / m["file"], map_location="cpu",
                            weights_only=False)
            model = build_model(int(m["config"]["unfreeze_last"]),
                                variant=m["config"]["variant"],
                                pool=m["config"].get("pool", "cls_mean"),
                                prior=bool(m["config"].get("prior", False))).to(dev)
            model.load_state_dict(ck["model"])
            check_fingerprint(model, dev, IMG, ck["fingerprint"], tag=f"{m['id']}: ")
            t_ready = time.time()
            p = predict_member(model, Cte, Mte, idx, dev, IMG, starts=starts)
            per_member.append({"id": m["id"], "ids": st_te, "pred": p,
                               "holdout": m.get("holdout")})
            fixed_s = t_ready - t0
            per_win_s = (time.time() - t_ready) / max(len(starts), 1)
            log(f"  {m['id']} fold {m['fold']}: predicted {len(idx)} studies over "
                f"{len(starts)} window(s) in {time.time() - t0:.0f}s")
            del model, ck
            gc.collect()
            if dev.type == "cuda":
                torch.cuda.empty_cache()
        del Cte, Mte
        gc.collect()

    # Rank em vez de probabilidade, porque a métrica lê ordem e dois members calibrados
    # de forma diferente de outro jeito teriam voz desigual. Todo member cobre todo
    # estudo de teste, então a média é sobre o mesmo conjunto toda vez e não precisa de
    # ponderação pra ser comparável; ponderar pelo holdout de um fold importaria pro
    # conjunto de teste uma diferença medida em algumas centenas de estudos de treino.
    all_ids = sorted({s for m in per_member for s in m["ids"]})
    pos = {s: i for i, s in enumerate(all_ids)}
    acc = np.zeros((len(all_ids), len(TARGETS)), np.float64)
    for m in per_member:
        r = pd.DataFrame(m["pred"]).rank(pct=True).to_numpy()
        acc[[pos[s] for s in m["ids"]]] += r
    acc /= max(len(per_member), 1)

    sub = write_submission(acc, all_ids, test_df, "submission.csv")
    log(f"submission.csv = rank mean of {len(per_member)} member(s); {sub.shape}; "
        f"nulls {int(sub[TARGETS].isna().sum().sum())}")
    return sub


def adopt_config_globals(cfg):
    """Aponta o caminho de pixels pro que um grupo de members foi ajustado."""
    global IMG, CACHE_IMG, GROUP, CACHE_SLICES, N_GROUP, CROP_MM, SLICE_BAND, RULES
    CACHE_IMG = IMG = int(cfg["img"])
    GROUP = int(cfg["group"])
    CACHE_SLICES = int(cfg["slices"])
    N_GROUP = max(CACHE_SLICES // GROUP, 1)
    CROP_MM = float(cfg["crop_mm"])
    SLICE_BAND = tuple(float(x) for x in cfg["band"])
    # As quatro decisões que mudam o que uma slice é. Um member ajustado sob uma leitura
    # e decodificado sob outra recebe pixels que seus pesos nunca viram, com todo shape
    # ainda concordando, então um nome não reconhecido é recusado em vez de receber um
    # valor padrão.
    rules = cfg.get("rules") or RULES_NATIVE
    unknown = {k: v for k, v in rules.items()
               if k not in RULES_NATIVE
               or v not in (RULES_NATIVE[k], RULES_LEGACY[k])}
    if unknown:
        raise WeightsError(f"the members record pixel rules this pipeline cannot "
                           f"reproduce: {unknown}")
    RULES = {**RULES_NATIVE, **rules}
    if [s[0] for s in SLOTS] != list(cfg["slots"]):
        raise WeightsError(
            f"the members were fitted on slots {cfg['slots']} and this pipeline defines "
            f"{[s[0] for s in SLOTS]}; a weight would be read against the wrong slot")


In [ ]:
def take_group(cache_rows, g):
    """Corta GROUP canais consecutivos das slices em cache."""
    return cache_rows[:, :, g * GROUP:(g + 1) * GROUP]


def augment(imgs):
    """Um pequeno jitter rígido e um scaling de intensidade, aplicados numa bag inteira
    de uma vez.

    Nenhum dos dois flips está disponível aqui, e por motivos diferentes. Um flip
    horizontal reintroduziria o eixo de ruído que a normalização de lateralidade
    removeu - desfaria, uma vez por batch, o que a passada de header foi feita pra
    estabelecer.

    Um flip vertical não é um eixo de ruído de jeito nenhum. Um joelho é adquirido numa
    orientação canônica, e nenhum estudo neste corpus se parece com o próprio espelho
    vertical. Uma augmentation deve cobrir direções ao longo das quais o label não muda;
    esta desloca o input pra fora da distribuição sobre a qual o encoder vai ser
    questionado, o que é uma coisa diferente. Onde um achado se situa no quadro também é
    informação, não ruído - um cisto de Baker é identificado por estar na fossa
    poplítea, não só pela aparência.

    O que sobra é jitter do qual nenhum label depende: alguns graus de rotação, alguns
    por cento de escala e translação. Isso ainda impede memorizar o enquadramento exato,
    que é pra isso que serve uma augmentation, enquanto deixa a anatomia onde estava.
    """
    # Uma bag chega como [study, slot, GROUP, IMG, IMG]: cinco eixos, não quatro. O warp
    # é uma operação 2-D, então os dois eixos iniciais são dobrados juntos e restaurados
    # depois - toda imagem de slot é uma aquisição independente e ganha seu próprio
    # jitter.
    lead = imgs.shape[:-3]
    x = imgs.reshape(-1, *imgs.shape[-3:]).float()
    n, dev = x.shape[0], x.device

    rot = (torch.rand(n, device=dev) - 0.5) * 2 * (AUG_ROT_DEG * np.pi / 180)
    # Só zoom pra dentro. O padding `border` repete a linha da borda pra fora, e a borda
    # deste corte é onde a fossa poplítea fica; zoom pra fora fabricaria tecido
    # exatamente onde um cisto de Baker é procurado.
    sc = 1.0 + torch.rand(n, device=dev) * AUG_SCALE
    tx = (torch.rand(n, device=dev) - 0.5) * 2 * AUG_SHIFT
    ty = (torch.rand(n, device=dev) - 0.5) * 2 * AUG_SHIFT
    cos, sin = torch.cos(rot) / sc, torch.sin(rot) / sc
    theta = torch.zeros(n, 2, 3, device=dev, dtype=torch.float32)
    theta[:, 0, 0], theta[:, 0, 1], theta[:, 0, 2] = cos, -sin, tx
    theta[:, 1, 0], theta[:, 1, 1], theta[:, 1, 2] = sin, cos, ty
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)

    scale = 1.0 + (torch.rand(n, 1, 1, 1, device=dev) - 0.5) * 2 * AUG_INTENSITY
    x = (x * scale).clamp(0, 255)
    return x.reshape(*lead, *x.shape[-3:]).to(imgs.dtype)


@torch.no_grad()
def predict(model, cache, mask, idx, dev, img_size=None):
    """Faz a média dos logits sobre os grupos de cada slot.

    O treino vê um grupo por vez, o que age como augmentation ao longo da pilha; a
    inferência faz a média sobre todos eles, então a predição não depende de qual grupo
    um único sorteio calhou de escolher. Onde o cache mantém um grupo por slot os dois
    coincidem.
    """
    model.eval()
    out = []
    for b in range(0, len(idx), EVAL_BATCH):
        sel = idx[b:b + EVAL_BATCH]
        m = torch.from_numpy(mask[sel]).to(dev)
        acc = None
        for g in range(N_GROUP):
            # Reunido um grupo por vez em vez de inteiro e depois cortado. Os dois são os
            # mesmos pixels, mas tirar um estudo inteiro do cache aloca toda slice que
            # ele mantém - a maioria das quais essa passada não vai olhar até uma
            # iteração posterior, quando já terão sido buscadas de novo. Medido sobre um
            # cache de doze slices, a diferença entre os dois é a diferença entre o passo
            # ser limitado por memória e ser limitado pelo encoder.
            rows = torch.from_numpy(np.ascontiguousarray(
                cache[sel, :, g * GROUP:(g + 1) * GROUP])).to(dev)
            with torch.autocast("cuda", enabled=dev.type == "cuda"):
                z = model(rows, m, img_size).float()
            acc = z if acc is None else acc + z
        out.append(torch.sigmoid(acc / N_GROUP).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, len(TARGETS)), np.float32)


def macro_auc(y, p):
    from sklearn.metrics import roc_auc_score
    return float(np.nanmean([roc_auc_score(y[:, j], p[:, j])
                             if len(set(y[:, j])) > 1 else np.nan
                             for j in range(y.shape[1])]))


In [ ]:
EXTRA_EPOCHS = 12
CONTINUE_LR_SCALE = 0.5  # LR menor que o treino original -- o member ja convergiu bastante


def main_continue_finetune():
    """Carrega o member mais fraco do pacote publicado (menor holdout
    gravado no manifest) e continua o treino dele por EXTRA_EPOCHS,
    usando o MESMO split (fold do member = grp==fold) pra comparacao
    direta e justa contra o holdout gravado. Salva um novo checkpoint só
    se bater o valor original -- não sobrescreve current member a troco de nada."""
    pkg = find_weights()
    if pkg is None:
        raise FileNotFoundError("pacote de pesos nao encontrado")
    man = json.loads((pkg / "manifest.json").read_text())
    members = man["members"]

    # Member alvo: o de MENOR holdout gravado (elo mais fraco do ensemble --
    # e onde continuar o treino tem mais chance de ajudar a media geral).
    target = min(members, key=lambda m: m.get("holdout") if m.get("holdout") is not None else 1.0)
    log(f"member alvo: {target['id']} (fold {target['fold']}, "
        f"epochs_done={target['epochs_done']}, holdout gravado={target['holdout']:.4f}, "
        f"annot gravado={target['annot']:.4f})")

    cfg = target["config"]
    adopt_config_globals(json.loads(target["pixel_group"]))
    # adopt_config_globals traz CACHE_SLICES=12 do pixel_group do member --
    # isso e o necessario pra TTA na INFERENCIA (janelas sobrepostas), mas
    # pro TREINO (so pega 1 grupo de GROUP consecutivos por passo) e 4x
    # mais cache do que precisa: (4407, 6, 12, 336, 336) = 33.4 GB, estourou
    # a RAM disponivel (29.4 GB) e matou o kernel na primeira tentativa.
    # Volta pro tamanho pequeno que o treino de fato usa (so 1 grupo).
    global CACHE_SLICES, N_GROUP
    CACHE_SLICES = GROUP
    N_GROUP = 1
    log(f"cache ajustado pro treino: {CACHE_SLICES} slices (nao os 12 do pixel_group, "
        f"que sao so pra TTA na inferencia)")

    train_df = pd.read_csv(ROOT / "train.csv")
    train_series = pd.read_csv(ROOT / "train_series.csv")
    plane_map = dict(zip(train_series["SeriesInstanceUID"], train_series["Anatomical_Plane"]))

    log("header pass: train")
    htr = annotate(walk("train_series"))
    log(f"  {len(htr)} train series")

    slots_tr = pick_slots(htr, plane_map)
    st_tr, Ctr, Mtr = build_cache(slots_tr, plane_map, lat_of(htr, "train "), "train")

    lab = read_labels(train_df)
    gold = train_df.set_index("StudyInstanceUID")[TARGETS]
    gold = gold[gold.notna().all(axis=1)]

    Y = np.zeros((len(st_tr), len(TARGETS)), np.float32)
    W = np.zeros_like(Y)
    for i, st in enumerate(st_tr):
        if st in gold.index:
            Y[i], W[i] = gold.loc[st].values, 3.0
        elif st in lab.index:
            r = lab.loc[st]
            Y[i] = r[TARGETS].values
            W[i] = 0.25 + 0.75 * r[[t + "__conf" for t in TARGETS]].values
    keep = np.where(W.sum(1) > 0)[0]
    log(f"supervised {len(keep)} of {len(st_tr)} studies (annotated {len(gold)})")

    # MESMO esquema de grupo (hash do texto do laudo) usado pra gerar os
    # holdouts gravados no manifest -- so muda QUAL grupo vira holdout
    # (fold do member), nao o esquema em si. Isso deixa nossa comparacao
    # diretamente comparavel ao valor gravado (ver PROGRESS.md, secao
    # metodologia de validacao -- holdout maior/baseado em pseudo-label,
    # nao os 58 gold).
    import hashlib
    rep = train_df.set_index("StudyInstanceUID")["Report"].fillna("")
    grp = np.array([int(hashlib.md5(rep.get(s, s).encode()).hexdigest()[:8], 16) % 5
                    for s in st_tr])
    fold = int(target["fold"])
    va = np.array([i for i in keep if grp[i] == fold])
    tr = np.array([i for i in keep if grp[i] != fold])
    log(f"train {len(tr)} / holdout {len(va)} studies (fold {fold}, mesmo split do member)")

    gpos = {s: i for i, s in enumerate(st_tr)}
    va_set = set(va.tolist())
    gi = np.array([gpos[s] for s in gold.index if s in gpos and gpos[s] in va_set])
    gold_y = gold.loc[[st_tr[i] for i in gi]].values.astype(int) if len(gi) else None
    yv = (Y[va] > 0.5).astype(int)
    log(f"annotation check: {len(gi)} of {len(gold)} annotated studies are in the holdout")

    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    ck = torch.load(pkg / target["file"], map_location=dev, weights_only=False)
    model = build_model(
        int(cfg["unfreeze_last"]), variant=cfg["variant"],
        pool=cfg.get("pool", "cls_mean"), prior=bool(cfg.get("prior", False)),
    ).to(dev)
    model.load_state_dict(ck["model"])
    check_fingerprint(model, dev, IMG, target["fingerprint"], tag=f"{target['id']} (antes de continuar): ")

    # Confirma o holdout ANTES de continuar treinando -- deve bater
    # (ou ficar bem perto) do valor gravado no manifest, validando que
    # carregamos certo e que estamos usando o mesmo split.
    pv0 = predict(model, Ctr, Mtr, va, dev, IMG)
    d0 = macro_auc(yv, pv0)
    log(f"holdout ANTES de continuar (deve bater ~{target['holdout']:.4f}): {d0:.4f}")

    opt = torch.optim.AdamW([
        {"params": [p for p in model.backbone.parameters() if p.requires_grad],
         "lr": LR_BACKBONE * CONTINUE_LR_SCALE},
        {"params": model.head.parameters(), "lr": LR_HEAD * CONTINUE_LR_SCALE},
    ], weight_decay=WEIGHT_DECAY)
    steps = max(EXTRA_EPOCHS * (len(tr) // BATCH_STUDIES), 1)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[LR_BACKBONE * CONTINUE_LR_SCALE, LR_HEAD * CONTINUE_LR_SCALE],
        total_steps=steps, pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")

    best, best_state, best_annot = d0, None, float("nan")
    for ep in range(EXTRA_EPOCHS):
        model.train()
        perm = np.random.permutation(tr)
        tot, nstep = 0.0, 0
        for b in range(0, len(perm) - BATCH_STUDIES + 1, BATCH_STUDIES):
            sel = perm[b:b + BATCH_STUDIES]
            rows = torch.from_numpy(Ctr[sel]).to(dev)
            g = int(torch.randint(N_GROUP, (1,)).item())
            imgs = augment(take_group(rows, g))
            m = torch.from_numpy(Mtr[sel]).to(dev)
            y = torch.from_numpy(Y[sel]).to(dev)
            w = torch.from_numpy(W[sel]).to(dev)
            with torch.autocast("cuda", enabled=dev.type == "cuda"):
                loss = (F.binary_cross_entropy_with_logits(
                    model(imgs, m, IMG), y, reduction="none") * w).mean()
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot += loss.item()
            nstep += 1

        pv = predict(model, Ctr, Mtr, va, dev, IMG)
        d = macro_auc(yv, pv)
        g_auc = float("nan")
        if gold_y is not None and len(gi):
            g_auc = macro_auc(gold_y, predict(model, Ctr, Mtr, gi, dev, IMG))
        log(f"  epoch {ep + 1}/{EXTRA_EPOCHS}  loss {tot / max(nstep, 1):.4f}"
            f"  holdout {d:.4f}  annot(n={len(gi)}) {g_auc:.4f}")

        if d > best:
            best, best_annot = d, g_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if time.time() - T0 > TIME_BUDGET:
            log("  time budget reached")
            break

    log(f"holdout ANTES: {d0:.4f}  |  MELHOR depois de continuar: {best:.4f}"
        f"  ({'MELHOROU' if best > d0 else 'nao melhorou'})")

    checkpoint_dir = Path("/kaggle/working/checkpoints")
    checkpoint_dir.mkdir(exist_ok=True, parents=True)
    if best_state is not None and best > d0:
        model.load_state_dict(best_state)
        new_fp = fingerprint(model, dev, IMG).tolist()
        out = {
            "model": model.state_dict(),
            "fingerprint": new_fp,
            "holdout": best,
            "annot": best_annot,
            "original_id": target["id"],
            "original_holdout": target["holdout"],
            "fold": target["fold"],
            "config": cfg,
        }
        out_path = checkpoint_dir / f"continued_{target['id']}.pt"
        torch.save(out, out_path)
        log(f"checkpoint melhorado salvo em {out_path}")
    else:
        log("nao salvou checkpoint novo -- continuar treinando nao melhorou "
            "o holdout desse member (ver PROGRESS.md pra essa decisao)")


In [ ]:
try:
    main_continue_finetune()
except Exception:
    traceback.print_exc()
    raise
log("done")
